In [1]:
#@title 0.1 Clone Repository & Install Dependencies
import os

# --- Set the data directory (adjust if running locally) ---
DATA_DIR = '/content/LLM_IP_assignment'   # <-- Colab default
# DATA_DIR = '.'   # <-- uncomment if running locally inside the repo

if not os.path.exists(DATA_DIR):
    !git clone https://github.com/matthewdelorenzo/LLM_IP_assignment.git {DATA_DIR}

os.chdir(DATA_DIR)
print('Working directory:', os.getcwd())

# Install Python dependencies
!pip install -q networkx openai

# Install Yosys (formal equivalence checker)
!apt-get install -y -q yosys

# Make the SIM binary executable
SIM_EXECUTABLE = os.path.join(DATA_DIR, 'sim', 'sim_text')
if os.path.exists(SIM_EXECUTABLE):
    os.chmod(SIM_EXECUTABLE, 0o755)
    print('SIM binary ready.')
else:
    print('WARNING: SIM binary not found at', SIM_EXECUTABLE)


Cloning into '/content/LLM_IP_assignment'...
remote: Enumerating objects: 484, done.
remote: Counting objects: 100% (484/484), done.
remote: Compressing objects: 100% (264/264), done.
remote: Total 484 (delta 123), reused 482 (delta 122), pack-reused 0 (from 0)
Receiving objects: 100% (484/484), 4.25 MiB | 7.45 MiB/s, done.
Resolving deltas: 100% (123/123), done.
Working directory: /content/LLM_IP_assignment
Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  at-spi2-core berkeley-abc gir1.2-atk-1.0 gir1.2-gtk-3.0
  gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data
  libatspi2.0-0 libgtk-3-0 libgtk-3-bin libgtk-3-common librsvg2-common
  libxcomposite1 libxtst6 python3-cairo python3-gi-cairo python3-numpy
  session-migration xdot
Suggested packages:
  gvfs python-numpy-doc python3-dev python3-pytest
The following NEW packages will be installed:
  at-spi2-core berkeley-abc gir1.2-a

In [2]:
#@title 0.2 Imports and Helper Functions
import sys, os, glob, pickle, copy, random, subprocess, time, re
import networkx as nx
import openai

# Add src/ directories to path
sys.path.insert(0, os.path.join(DATA_DIR, 'src'))
sys.path.insert(0, os.path.join(DATA_DIR, 'src_ablation_study_wo_SolB'))

# Core functions from original codebase
from characterize_circuit import (
    characterize_generic_netlist,
    create_LLM_circuit_query_template,
    is_gate_line,
    convert_circuit_to_Boolean_format,
    convert_circuit_from_Boolean_format,
)
from circuit_decomposer import (
    parse_expression,
    decompose_expression,
    final_decompose_expression,
)
from sim_evaluation import evaluate_sim_text
from config import DATASET_PATH, SIM_DIR

# Helper functions for the workshop
def evaluate_sim(orig_contents, pirated_contents, label='test'):
    """Save circuits to temp files and run SIM text comparison."""
    os.makedirs('tmp_eval', exist_ok=True)
    orig_path = f'tmp_eval/orig_{label}.v'
    pirated_path = f'tmp_eval/pirated_{label}.v'

    with open(orig_path, 'w') as f:
        f.write(orig_contents)
    with open(pirated_path, 'w') as f:
        f.write(pirated_contents)

    try:
        result = evaluate_sim_text(orig_path, pirated_path)
        if result.returncode != 0:
            print(f'SIM error (exit code {result.returncode}): {result.stderr}')
            return None
        if 'consists for' in result.stdout:
            score = float(
                result.stdout.split('\n')[-2]
                .split('consists for')[-1]
                .split('%')[0].strip()
            ) / 100.0
            return score
        return 0.0
    except Exception as e:
        print(f'SIM error: {e}')
        return None


def _strip_code_fences(text):
    """Remove markdown code fences from LLM responses."""
    if '```' in text:
        parts = text.split('```')
        if len(parts) >= 2:
            content = parts[1]
            lines = content.split('\n')
            if lines and lines[0].strip().lower() in ('', 'verilog', 'v', 'sv', 'systemverilog'):
                content = '\n'.join(lines[1:])
            return content.strip()
    return text


def _fix_verilog_declarations(verilog):
    """
    Fix common LLM Verilog output issues:
    - Add missing semicolons to input/output/wire/reg declaration lines
      that end a statement but are missing the terminating ';'
    """
    _DECL_KW = re.compile(r'^\s*(input|output|inout|wire|reg)\b')
    fixed_lines = []
    lines = verilog.splitlines()
    for i, line in enumerate(lines):
        stripped = line.rstrip()
        if _DECL_KW.match(stripped):
            # Only add ';' if the line doesn't already end with ';' or ','
            # and it's not a continuation line (next line also starts a declaration
            # or is blank / endmodule)
            if stripped and not stripped.endswith(';') and not stripped.endswith(','):
                stripped = stripped + ';'
        fixed_lines.append(stripped)
    return '\n'.join(fixed_lines)


def _get_module_name(verilog):
    """Extract the top module name from a Verilog string."""
    for line in verilog.splitlines():
        m = re.match(r'\s*module\s+(\w+)', line)
        if m:
            return m.group(1)
    return None


def check_functional_equivalence(orig_verilog, transformed_verilog, label='test'):
    """
    Formal equivalence check using Yosys miter circuit + SAT solver.

    Runs the following Yosys flow:
        read_verilog
        read_verilog
        prep; proc; opt; memory;
        clk2fflogic;
        miter -equiv -flatten   miter
        sat -seq 50 -verify -prove trigger 0 -show-all \
            -show-inputs -show-outputs -set-init-zero miter

    Returns: (is_equivalent: bool | None, details: str)
        True  — formally proven equivalent
        False — counterexample found (not equivalent)
        None  — Yosys error or inconclusive result
    """
    os.makedirs('tmp_eval', exist_ok=True)

    # Strip markdown fences and fix common LLM Verilog formatting issues
    orig_clean = _fix_verilog_declarations(_strip_code_fences(orig_verilog))
    trans_clean = _fix_verilog_declarations(_strip_code_fences(transformed_verilog))

    orig_module = _get_module_name(orig_clean)
    gen_module  = _get_module_name(trans_clean)

    if not orig_module:
        return None, "Could not extract module name from original Verilog"
    if not gen_module:
        return None, "Could not extract module name from transformed Verilog"

    # Rename both modules to unique names so Yosys never sees a name clash
    gold_name = 'gold_circuit'
    gate_name = 'gate_circuit'

    gold_verilog = re.sub(
        r'\bmodule\s+' + re.escape(orig_module) + r'\b',
        f'module {gold_name}', orig_clean, count=1
    )
    gate_verilog = re.sub(
        r'\bmodule\s+' + re.escape(gen_module) + r'\b',
        f'module {gate_name}', trans_clean, count=1
    )

    truth_path  = os.path.abspath(f'tmp_eval/truth_{label}.v')
    gen_path    = os.path.abspath(f'tmp_eval/gen_{label}.v')
    script_path = os.path.abspath(f'tmp_eval/eq_{label}.ys')

    with open(truth_path, 'w') as f:
        f.write(gold_verilog)
    with open(gen_path, 'w') as f:
        f.write(gate_verilog)

    yosys_script = (
        f"read_verilog {truth_path}\n"
        f"read_verilog {gen_path}\n"
        "prep; proc; opt; memory;\n"
        "clk2fflogic;\n"
        f"miter -equiv -flatten {gate_name} {gold_name} miter\n"
        "sat -seq 50 -verify -prove trigger 0 "
        "-show-all -show-inputs -show-outputs -set-init-zero miter\n"
    )
    with open(script_path, 'w') as f:
        f.write(yosys_script)

    print(f"🔍 Running Yosys equivalence check for '{label}'...")
    try:
        result = subprocess.run(
            ['yosys', '-s', script_path],
            capture_output=True, text=True, timeout=180
        )
        output = result.stdout + result.stderr

        if 'no model found: SUCCESS' in output:
            return True, "Circuits formally verified equivalent (Yosys miter+SAT)"
        elif 'proof did fail' in output:
            return False, "Circuits NOT equivalent — Yosys SAT found a counterexample"
        elif result.returncode != 0:
            err_lines = [l.strip() for l in output.splitlines() if 'ERROR' in l or 'Error' in l]
            err_msg = '; '.join(err_lines[:3]) if err_lines else output[-400:]
            return None, f"Yosys error: {err_msg}"
        else:
            return None, f"Yosys result inconclusive (exit {result.returncode})"

    except subprocess.TimeoutExpired:
        return None, "Yosys equivalence check timed out (180 s)"
    except FileNotFoundError:
        return None, "Yosys not found — run: !apt-get install -y yosys  (or re-run cell 3)"
    except Exception as e:
        return None, f"Equivalence check error: {e}"


def get_mapped_circuit(orig_file_contents, cached_circuit_mapping, mapping_strategy, rank=0):
    """
    Replace every gate in the original circuit with the LLM's cached rephrase.
    From evaluate_piracy_using_cached_mapping_multiprocessing_v2.py in the original codebase.
    """
    new_lines = []
    new_wires = []
    lines = orig_file_contents.split(';')
    lines = [line.strip() for line in lines if line.strip()]
    lines = [line.replace('\n', ' ') for line in lines]
    for line in lines:
        if line.startswith(('module ', 'input ', 'output ', 'wire ', 'endmodule')):
            new_lines.append(line)
            continue
        if is_gate_line(line):
            keep_line_same = False
            gate_type = line.split()[0] + '_' + str(line.count(','))
            if gate_type not in cached_circuit_mapping or len(cached_circuit_mapping[gate_type]) == 0:
                new_lines.append(line)
                continue
            if len(list(cached_circuit_mapping[gate_type].keys())) > 0:
                line = line.replace('(', ' ( ')
                line = line.replace(')', ' ) ')
                gate_type = line.split(' ')[0] + '_' + str(line.count(','))
                gate_name = line.split()[1].strip()
                op = line.split()[3].strip(',').strip()
                num_ips = line.count(',')
                ips = [line.split()[4+j].strip().strip(',').strip() for j in range(num_ips)]
                if mapping_strategy == 'random':
                    target_mapping_gate = random.choice(list(cached_circuit_mapping[gate_type].keys()))
                else:
                    if mapping_strategy.split('_')[0] == gate_type.split('_')[0].upper():
                        keep_line_same = True
                    elif mapping_strategy in list(cached_circuit_mapping[gate_type].keys()):
                        target_mapping_gate = copy.deepcopy(mapping_strategy)
                    else:
                        target_mapping_gate = random.choice(list(cached_circuit_mapping[gate_type].keys()))
                if keep_line_same:
                    new_lines.append(line)
                else:
                    mapped_circuit_str = cached_circuit_mapping[gate_type][target_mapping_gate][0]
                    nets = []
                    mapped_lines = mapped_circuit_str.split('\n')
                    for l in mapped_lines:
                        l = l.strip().strip(';')
                        tmp_inputs = l.split('(')[-1].split(')')[0].split(',')
                        tmp_inputs = [inp.strip() for inp in tmp_inputs]
                        tmp_output = l.split('=')[0].strip()
                        nets.extend(tmp_inputs)
                        nets.append(tmp_output)
                    nets = list(set(nets))
                    inter_net_cnt = 0
                    for net in nets:
                        if net == 'Y':
                            for pat in ['Y = ', 'Y= ', 'Y =', 'Y=']:
                                if pat in mapped_circuit_str:
                                    mapped_circuit_str = mapped_circuit_str.replace(pat, op + pat[1:])
                                    break
                        elif net in ['A' + str(i) for i in range(1, 50)]:
                            idx = int(net[1:]) - 1
                            for pat in ['(' + net + ',', ' ' + net + ',', ' ' + net + ')', '(' + net + ')']:
                                repl = pat.replace(net, ips[idx])
                                mapped_circuit_str = mapped_circuit_str.replace(pat, repl)
                        else:
                            wire_name = gate_name + '_inter_net_' + str(inter_net_cnt)
                            inter_net_cnt += 1
                            for pat in ['(' + net + ',', ' ' + net + ',', ' ' + net + ')', '(' + net + ')']:
                                mapped_circuit_str = mapped_circuit_str.replace(pat, pat.replace(net, wire_name))
                            if net + ' = ' in mapped_circuit_str:
                                mapped_circuit_str = mapped_circuit_str.replace(net + ' = ', wire_name + ' = ')
                            new_wires.append(wire_name)
                    mapped_lines = mapped_circuit_str.split('\n')
                    cnt = 0
                    for l in mapped_lines:
                        l = l.strip().strip(';')
                        if not l:
                            continue
                        tmp_inputs = l.split('(')[-1].split(')')[0].split(',')
                        tmp_inputs = [inp.strip() for inp in tmp_inputs]
                        tmp_output = l.split('=')[0].strip()
                        tmp_gate_type = l.split('=')[-1].split('(')[0].strip()
                        if tmp_gate_type in ['AND', 'OR', 'NAND', 'NOR'] and len(tmp_inputs) == 1:
                            tmp_inputs = [tmp_inputs[0], tmp_inputs[0]]
                        new_str = tmp_gate_type.lower() + ' ' + gate_name + '_' + str(cnt) + ' ( ' + tmp_output + ', ' + ', '.join(tmp_inputs) + ' )'
                        cnt += 1
                        new_lines.append(new_str)
            else:
                new_lines.append(line)
        else:
            new_lines.append(line)
    if new_wires:
        for i in range(len(new_lines)):
            if new_lines[i].split()[0] == 'wire':
                new_lines[i] = new_lines[i] + ';\nwire ' + ', '.join(new_wires)
                break
    return ';\n'.join(new_lines)


def get_circuit_from_response(response):
    """Parse LLM response to extract gate equations (from original code)."""
    LLM_circuit = ""
    if "```" in response:
        lines = response.split("```")[1].split("\n")
        lines = [l for l in lines if l != '']
    elif "=" in response:
        lines = response.split("\n")
    else:
        return "Incorrect format"

    for line in lines:
        if ("=" in line) and (not line.startswith("wire ")):
            tmp_expression = line.split("=")[-1].strip().strip(";")
            if any(f"{g} " in tmp_expression for g in ["AND", "OR", "NAND", "NOR", "XOR", "XNOR"]):
                return "Incorrect format"
            tmp_op_net = line.split("=")[0].strip()
            decomposed = final_decompose_expression(tmp_expression, tmp_op_net)
            LLM_circuit += decomposed + "\n"

    if not LLM_circuit.strip():
        return "Incorrect format"
    return LLM_circuit.rstrip("\n")


SIM_THRESHOLD = 0.3

print('✅ All imports successful. Using Yosys formal equivalence checking (miter + SAT).')
print(f'SIM detection threshold: {SIM_THRESHOLD}')


✅ All imports successful. Using Yosys formal equivalence checking (miter + SAT).
SIM detection threshold: 0.3


In [ ]:
#@title 0.3 Load Circuit & Setup OpenAI Client

# Load the test circuit
DESIGN = 'C432'
VARIANT = 'c432-CS320'

circuit_path = os.path.join(DATASET_PATH, DESIGN, VARIANT, 'topModule.v')
with open(circuit_path, 'r') as f:
    orig_file_contents = f.read()

gate_counts = characterize_generic_netlist(orig_file_contents)
total_gates = sum(gate_counts.values())

print(f'Loaded circuit: {DESIGN}/{VARIANT}')
print(f'Total gates: {total_gates}')
print(f'Gate types: {dict(gate_counts)}')
print(f'Circuit size: {len(orig_file_contents)} characters')

# Show first 400 chars of the circuit
print(f'\nFirst 400 characters of circuit:')
print(orig_file_contents[:400] + '...')

# Set up OpenAI client for LLM experiments
# TODO: Set your OpenAI API key here
OPENAI_API_KEY = ""   # <-- Students: paste your API key

if not OPENAI_API_KEY:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

if OPENAI_API_KEY:
    openai.api_key = OPENAI_API_KEY
    print("\n✅ OpenAI API configured")
else:
    print("\n⚠️  WARNING: No API key set. You'll need this for live LLM querying.")
    print("Set OPENAI_API_KEY above or use the environment variable.")

Loaded circuit: C432/c432-CS320
Total gates: 212
Gate types: {'nand_2': 64, 'not_1': 60, 'xor_2': 32, 'nor_2': 19, 'xnor_2': 18, 'nand_4': 14, 'and_9': 3, 'and_8': 1, 'nand_3': 1}
Circuit size: 10789 characters

First 400 characters of circuit:
module top (N1,N4,N8,N11,N14,N17,N21,N24,N27,N30,
             N34,N37,N40,N43,N47,N50,N53,N56,N60,N63,
             N66,N69,N73,N76,N79,N82,N86,N89,N92,N95,
             N99,N102,N105,N108,N112,N115,N223,N329,N370,N421,
                  N430,N431,N432,
        keyIn_0_0, keyIn_0_1, keyIn_0_2, keyIn_0_3, keyIn_0_4,
        keyIn_0_5, keyIn_0_6, keyIn_0_7, keyIn_0_8, keyIn_0_9,
        keyIn_0_10,...

✅ OpenAI API configured


In [16]:
#@title Task 1: Direct Verilog Rewriting Implementation

# TODO: Students can modify these settings
TASK1_TARGET_GATES = "NOR"  # Try: "NOR", "NAND", "AND and NOT"
TASK1_MODEL = "gpt-3.5-turbo-16k"  # Try: "gpt-4"

# TODO: Students should improve this prompt template

task1_prompt = f"""
Rewrite the following Verilog circuit using only {TASK1_TARGET_GATES}-based structural logic.

Strict requirements:
1. Preserve the exact same module name.
2. Preserve the exact same input/output port list and port order.
3. Preserve the exact same functionality.
4. Output only complete Verilog code. Do not include any explanation, comments, markdown, or extra text.
5. End the output with a valid 'endmodule'.
6. Do not omit any logic.
7. Keep all internal signals consistently declared and connected.
8. The rewritten circuit must be syntactically valid and compilable.
9. Do not use assign expressions with unsupported operators if they are not in the target gate set.
10. If inversion is needed, implement it structurally using the target gate set.

Gate-set restriction:
- Use only {TASK1_TARGET_GATES} gates to implement the logic.
- Do not directly use AND, OR, NAND, XOR, XNOR, BUF, or NOT unless they are constructed structurally from {TASK1_TARGET_GATES}.
- Preserve the top-level interface exactly.

Return format:
- Return only the full rewritten Verilog module.
- No prose before or after the code.

Original circuit:
{orig_file_contents}
"""

print("Task 1 Prompt Summary:")
print("=" * 50)
print(f"Target gates: {TASK1_TARGET_GATES}")
print(f"Model: {TASK1_MODEL}")
print(f"Prompt length: {len(task1_prompt)} characters")
print("\nFirst 400 chars of prompt:")
print(task1_prompt[:400] + "...")

# Initialize result variables
task1_result = None
task1_sim_score = None
task1_evades = False
task1_functionally_correct = None

# Task 1 Execution
if OPENAI_API_KEY:
    print("\n" + "=" * 50)
    print("EXECUTING TASK 1...")
    print("=" * 50)

    try:
        response = openai.chat.completions.create(
            model=TASK1_MODEL,
            messages=[{"role": "user", "content": task1_prompt}],
            temperature=0.3,
            max_tokens=4000
        )
        task1_result = response.choices[0].message.content

        print(f"✅ LLM Response received ({len(task1_result)} chars)")
        print("First 400 chars of response:")
        print(task1_result[:400] + "...")

        # Evaluate functional correctness
        print("\n🔍 Checking functional equivalence...")
        equiv_result, equiv_details = check_functional_equivalence(
            orig_file_contents, task1_result, "task1"
        )
        task1_functionally_correct = equiv_result
        print(f"Functional check: {equiv_details}")

        # Evaluate with SIM
        print("\n🔍 Evaluating with SIM detector...")
        task1_sim_score = evaluate_sim(orig_file_contents, task1_result, "task1")
        task1_evades = task1_sim_score is not None and task1_sim_score < SIM_THRESHOLD

        print(f"\n📊 TASK 1 RESULTS:")
        print(f"Functional Correctness: {'✅ YES' if task1_functionally_correct else '❌ NO' if task1_functionally_correct is False else '⚠️ UNKNOWN'}")
        print(f"SIM Score: {task1_sim_score:.4f}" if task1_sim_score else "SIM Score: Error")
        print(f"Evades Detection: {'✅ YES' if task1_evades else '❌ NO'}")
        print(f"Model Used: {TASK1_MODEL}")
        print(f"Target Gates: {TASK1_TARGET_GATES}")

        # Overall success requires BOTH functional correctness AND SIM evasion
        overall_success = task1_functionally_correct and task1_evades
        print(f"Overall Success: {'✅ YES' if overall_success else '❌ NO'}")

    except Exception as e:
        print(f"❌ Task 1 failed: {e}")
        task1_result = None
        task1_sim_score = None
        task1_evades = False
        task1_functionally_correct = None
else:
    print("⚠️ Skipping Task 1 execution (no API key)")

Task 1 Prompt Summary:
Target gates: NOR
Model: gpt-3.5-turbo-16k
Prompt length: 11873 characters

First 400 chars of prompt:

Rewrite the following Verilog circuit using only NOR-based structural logic.

Strict requirements:
1. Preserve the exact same module name.
2. Preserve the exact same input/output port list and port order.
3. Preserve the exact same functionality.
4. Output only complete Verilog code. Do not include any explanation, comments, markdown, or extra text.
5. End the output with a valid 'endmodule'.
6. ...

EXECUTING TASK 1...
✅ LLM Response received (8493 chars)
First 400 chars of response:
module top (N1,N4,N8,N11,N14,N17,N21,N24,N27,N30,
             N34,N37,N40,N43,N47,N50,N53,N56,N60,N63,
             N66,N69,N73,N76,N79,N82,N86,N89,N92,N95,
             N99,N102,N105,N108,N112,N115,N223,N329,N370,N421,
                  N430,N431,N432,
        keyIn_0_0, keyIn_0_1, keyIn_0_2, keyIn_0_3, keyIn_0_4,
        keyIn_0_5, keyIn_0_6, keyIn_0_7, keyIn_0_8, keyIn_0_9,
  

In [12]:
from IPython.display import display, Markdown

analysis_md = """
## Task 1 Analysis

**Model used:** gpt-3.5-turbo-16k
**SIM Score:** 0.2200
**Success/Failure:** Partial success

### Key Insights
Direct Verilog rewriting is difficult because the LLM must preserve syntax, connectivity, and functionality in one shot. Although the output evaded SIM detection, it was not syntactically valid.
"""

display(Markdown(analysis_md))


## Task 1 Analysis

**Model used:** gpt-3.5-turbo-16k  
**SIM Score:** 0.2200  
**Success/Failure:** Partial success  

### Key Insights
Direct Verilog rewriting is difficult because the LLM must preserve syntax, connectivity, and functionality in one shot. Although the output evaded SIM detection, it was not syntactically valid.


In [14]:
#@title Task 1: Direct Verilog Rewriting Implementation

# TODO: Students can modify these settings
TASK1_TARGET_GATES = "NOR"  # Try: "NOR", "NAND", "AND and NOT"
TASK1_MODEL = "gpt-3.5-turbo-16k"  # Try: "gpt-4"

# TODO: Students should improve this prompt template
task1_prompt = f"""
Rewrite the following Verilog circuit using only {TASK1_TARGET_GATES} gates.

Strict requirements:
1. Preserve the exact same module name.
2. Preserve the exact same port list, names, directions, and order.
3. Preserve the exact same logic functionality.
4. Return only complete Verilog code.
5. Do not include any explanation, comments, markdown, or natural language.
6. The output must be syntactically valid and compilable.
7. End with 'endmodule'.
8. Do not omit any part of the circuit.
9. Keep wire declarations correct and consistent.
10. Use only {TASK1_TARGET_GATES} gates to realize all logic.
11. Do not directly use AND, OR, NAND, XOR, XNOR, BUF, or NOT unless they are structurally built from {TASK1_TARGET_GATES}.
12. If inversion is required, build it using {TASK1_TARGET_GATES}.
13. Ensure all parentheses, semicolons, declarations, and module boundaries are complete.

Return only the rewritten Verilog module.

Original circuit:
{orig_file_contents}
"""


print("Task 1 Prompt Summary:")
print("=" * 50)
print(f"Target gates: {TASK1_TARGET_GATES}")
print(f"Model: {TASK1_MODEL}")
print(f"Prompt length: {len(task1_prompt)} characters")
print("\nFirst 400 chars of prompt:")
print(task1_prompt[:400] + "...")

# Initialize result variables
task1_result = None
task1_sim_score = None
task1_evades = False
task1_functionally_correct = None

# Task 1 Execution
if OPENAI_API_KEY:
    print("\n" + "=" * 50)
    print("EXECUTING TASK 1...")
    print("=" * 50)

    try:
        response = openai.chat.completions.create(
            model=TASK1_MODEL,
            messages=[{"role": "user", "content": task1_prompt}],
            temperature=0.3,
            max_tokens=4000
        )
        task1_result = response.choices[0].message.content

        print(f"✅ LLM Response received ({len(task1_result)} chars)")
        print("First 400 chars of response:")
        print(task1_result[:400] + "...")

        # Evaluate functional correctness
        print("\n🔍 Checking functional equivalence...")
        equiv_result, equiv_details = check_functional_equivalence(
            orig_file_contents, task1_result, "task1"
        )
        task1_functionally_correct = equiv_result
        print(f"Functional check: {equiv_details}")

        # Evaluate with SIM
        print("\n🔍 Evaluating with SIM detector...")
        task1_sim_score = evaluate_sim(orig_file_contents, task1_result, "task1")
        task1_evades = task1_sim_score is not None and task1_sim_score < SIM_THRESHOLD

        print(f"\n📊 TASK 1 RESULTS:")
        print(f"Functional Correctness: {'✅ YES' if task1_functionally_correct else '❌ NO' if task1_functionally_correct is False else '⚠️ UNKNOWN'}")
        print(f"SIM Score: {task1_sim_score:.4f}" if task1_sim_score else "SIM Score: Error")
        print(f"Evades Detection: {'✅ YES' if task1_evades else '❌ NO'}")
        print(f"Model Used: {TASK1_MODEL}")
        print(f"Target Gates: {TASK1_TARGET_GATES}")

        # Overall success requires BOTH functional correctness AND SIM evasion
        overall_success = task1_functionally_correct and task1_evades
        print(f"Overall Success: {'✅ YES' if overall_success else '❌ NO'}")

    except Exception as e:
        print(f"❌ Task 1 failed: {e}")
        task1_result = None
        task1_sim_score = None
        task1_evades = False
        task1_functionally_correct = None
else:
    print("⚠️ Skipping Task 1 execution (no API key)")

Task 1 Prompt Summary:
Target gates: NOR
Model: gpt-3.5-turbo-16k
Prompt length: 11670 characters

First 400 chars of prompt:

Rewrite the following Verilog circuit using only NOR gates.

Strict requirements:
1. Preserve the exact same module name.
2. Preserve the exact same port list, names, directions, and order.
3. Preserve the exact same logic functionality.
4. Return only complete Verilog code.
5. Do not include any explanation, comments, markdown, or natural language.
6. The output must be syntactically valid and c...

EXECUTING TASK 1...
✅ LLM Response received (8472 chars)
First 400 chars of response:
module top (N1,N4,N8,N11,N14,N17,N21,N24,N27,N30,
             N34,N37,N40,N43,N47,N50,N53,N56,N60,N63,
             N66,N69,N73,N76,N79,N82,N86,N89,N92,N95,
             N99,N102,N105,N108,N112,N115,N223,N329,N370,N421,
                  N430,N431,N432,
        keyIn_0_0, keyIn_0_1, keyIn_0_2, keyIn_0_3, keyIn_0_4,
        keyIn_0_5, keyIn_0_6, keyIn_0_7, keyIn_0_8, keyIn_0_9,
  

In [15]:
from IPython.display import display, Markdown

analysis_md = """
## Task 1 Analysis

**Model used:** gpt-3.5-turbo-16k
**SIM Score:** 0.2200
**Success/Failure:** Partial success

### Key Insights
Direct Verilog rewriting is difficult even if i tried gpt4 it still didnt make sense because the LLM must preserve syntax, connectivity, and functionality in one shot. Although the output evaded SIM detection, it was not syntactically valid.
"""

display(Markdown(analysis_md))


## Task 1 Analysis

**Model used:** gpt-3.5-turbo-16k  
**SIM Score:** 0.2200  
**Success/Failure:** Partial success  

### Key Insights
Direct Verilog rewriting is difficult even if i tried gpt4 it still didnt make sense because the LLM must preserve syntax, connectivity, and functionality in one shot. Although the output evaded SIM detection, it was not syntactically valid.


In [17]:
#@title Task 2: Boolean Format Rewriting Implementation

# Convert circuit to Boolean format (this makes it easier for LLM)
circ_in_Boolean_format, remaining_orig_lines = convert_circuit_to_Boolean_format(orig_file_contents)

print("Boolean Format Conversion:")
print("=" * 30)
print(f"Boolean equations: {len([l for l in circ_in_Boolean_format.splitlines() if l.strip()])} lines")
print(f"Remaining structural lines: {len(remaining_orig_lines.splitlines())} lines")
print("\nFirst 300 chars of Boolean format:")
print(circ_in_Boolean_format[:300] + "...")

# TODO: Students configure task 2
TASK2_TARGET_GATES = "NOR"  # Try: "NOR", "NAND", "AND and NOT"
TASK2_MODEL = "gpt-3.5-turbo-16k"

task2_prompt = f"""
Rewrite the following Boolean equations using ONLY {TASK2_TARGET_GATES} gates.

Strict requirements:
1. Preserve the exact same functionality.
2. Keep all original variable names exactly the same wherever they appear.
3. Output ONLY rewritten Boolean equations.
4. Do NOT include any explanation, comments, markdown, headings, or extra text.
5. Each output line must contain exactly one assignment in this format:
   variable = GATE(arg1, arg2)
6. Use only {TASK2_TARGET_GATES} gates on the right-hand side.
7. Do NOT use NOT, AND, OR, NAND, XOR, or XNOR directly unless they are structurally implemented using {TASK2_TARGET_GATES}.
8. If inversion is needed, implement it using {TASK2_TARGET_GATES}.
9. Do NOT skip any equation.
10. Do NOT change output names.
11. If intermediate variables are absolutely necessary, define them clearly using the same assignment format.
12. Every right-hand side must be a valid gate call. No plain expressions, no prose, no blank explanations.

Important formatting rules:
- Return only equations.
- No numbering.
- No bullet points.
- No surrounding code fences.
- No text before the first equation.
- No text after the last equation.

Boolean equations to rewrite:
{circ_in_Boolean_format}
"""

print(f"\nTask 2 Configuration:")
print(f"Target gates: {TASK2_TARGET_GATES}")
print(f"Model: {TASK2_MODEL}")
print(f"Prompt length: {len(task2_prompt)} characters")

# Initialize results
task2_result = None
task2_sim_score = None
task2_evades = False
task2_functionally_correct = None

# Task 2 Execution
if OPENAI_API_KEY:
    print("\n" + "=" * 50)
    print("EXECUTING TASK 2...")
    print("=" * 50)

    try:
        response = openai.chat.completions.create(
            model=TASK2_MODEL,
            messages=[{"role": "user", "content": task2_prompt}],
            temperature=0.3,
            max_tokens=4000
        )
        llm_boolean_response = response.choices[0].message.content

        print(f"✅ LLM Response received ({len(llm_boolean_response)} chars)")
        print("First 300 chars of Boolean response:")
        print(llm_boolean_response[:300] + "...")

        # Convert back to Verilog format
        print("\n🔄 Converting back to Verilog format...")
        task2_result = convert_circuit_from_Boolean_format(llm_boolean_response, remaining_orig_lines)

        print(f"✅ Converted back to Verilog ({len(task2_result)} chars)")

        # Check functional equivalence
        print("\n🔍 Checking functional equivalence...")
        equiv_result, equiv_details = check_functional_equivalence(
            orig_file_contents, task2_result, "task2"
        )
        task2_functionally_correct = equiv_result
        print(f"Functional check: {equiv_details}")

        # Evaluate with SIM
        print("\n🔍 Evaluating with SIM detector...")
        task2_sim_score = evaluate_sim(orig_file_contents, task2_result, "task2")
        task2_evades = task2_sim_score is not None and task2_sim_score < SIM_THRESHOLD

        print(f"\n📊 TASK 2 RESULTS:")
        print(f"Functional Correctness: {'✅ YES' if task2_functionally_correct else '❌ NO' if task2_functionally_correct is False else '⚠️ UNKNOWN'}")
        print(f"SIM Score: {task2_sim_score:.4f}" if task2_sim_score else "SIM Score: Error")
        print(f"Evades Detection: {'✅ YES' if task2_evades else '❌ NO'}")
        print(f"Model Used: {TASK2_MODEL}")
        print(f"Target Gates: {TASK2_TARGET_GATES}")

        # Overall success requires BOTH functional correctness AND SIM evasion
        overall_success = task2_functionally_correct and task2_evades
        print(f"Overall Success: {'✅ YES' if overall_success else '❌ NO'}")

    except Exception as e:
        print(f"❌ Task 2 failed: {e}")
        task2_result = None
        task2_sim_score = None
        task2_evades = False
        task2_functionally_correct = None
else:
    print("⚠️ Skipping Task 2 execution (no API key)")

Boolean Format Conversion:
Boolean equations: 212 lines
Remaining structural lines: 13 lines

First 300 chars of Boolean format:
KeyWire_0[0] = NOT(N1)
N118 = XNOR(keyIn_0_0, KeyWire_0[0])
N119 = NOT(N4)
KeyWire_0[1] = NOT(N11)
N122 = XNOR(keyIn_0_1, KeyWire_0[1])
N123 = NOT(N17)
KeyWire_0[2] = NOT(N24)
KeyNOTWire_0[0] = XNOR(keyIn_0_2, KeyWire_0[2])
N126 = NOT(KeyNOTWire_0[0])
N127 = NOT(N30)
KeyWire_0[3] = NOT(N37)
N130 = X...

Task 2 Configuration:
Target gates: NOR
Model: gpt-3.5-turbo-16k
Prompt length: 7207 characters

EXECUTING TASK 2...
✅ LLM Response received (8454 chars)
First 300 chars of Boolean response:
KeyWire_0[0] = NOR(N1, N1)
N118 = NOR(NOR(NOR(keyIn_0_0, NOR(N1, N1)), NOR(N1, N1)), NOR(NOR(keyIn_0_0, NOR(N1, N1)), NOR(N1, N1)))
N119 = NOR(NOR(NOR(N4, N4), NOR(N4, N4)), NOR(NOR(N4, N4), NOR(N4, N4)))
KeyWire_0[1] = NOR(NOR(N11, N11), NOR(N11, N11))
N122 = NOR(NOR(NOR(keyIn_0_1, NOR(N11, N11)), ...

🔄 Converting back to Verilog format...
✅ Converted back to Verilog (66

In [18]:
#@title Task 2: Boolean Format Rewriting Implementation

# Convert circuit to Boolean format (this makes it easier for LLM)
circ_in_Boolean_format, remaining_orig_lines = convert_circuit_to_Boolean_format(orig_file_contents)

print("Boolean Format Conversion:")
print("=" * 30)
print(f"Boolean equations: {len([l for l in circ_in_Boolean_format.splitlines() if l.strip()])} lines")
print(f"Remaining structural lines: {len(remaining_orig_lines.splitlines())} lines")
print("\nFirst 300 chars of Boolean format:")
print(circ_in_Boolean_format[:300] + "...")

# TODO: Students configure task 2
TASK2_TARGET_GATES = "NOR"  # Try: "NOR", "NAND", "AND and NOT"
TASK2_MODEL = "gpt-4"

task2_prompt = f"""
Rewrite the following Boolean equations using ONLY {TASK2_TARGET_GATES} gates.

Strict requirements:
1. Preserve the exact same functionality.
2. Keep all original variable names exactly the same wherever they appear.
3. Output ONLY rewritten Boolean equations.
4. Do NOT include any explanation, comments, markdown, headings, or extra text.
5. Each output line must contain exactly one assignment in this format:
   variable = GATE(arg1, arg2)
6. Use only {TASK2_TARGET_GATES} gates on the right-hand side.
7. Do NOT use NOT, AND, OR, NAND, XOR, or XNOR directly unless they are structurally implemented using {TASK2_TARGET_GATES}.
8. If inversion is needed, implement it using {TASK2_TARGET_GATES}.
9. Do NOT skip any equation.
10. Do NOT change output names.
11. If intermediate variables are absolutely necessary, define them clearly using the same assignment format.
12. Every right-hand side must be a valid gate call. No plain expressions, no prose, no blank explanations.

Important formatting rules:
- Return only equations.
- No numbering.
- No bullet points.
- No surrounding code fences.
- No text before the first equation.
- No text after the last equation.

Boolean equations to rewrite:
{circ_in_Boolean_format}
"""

print(f"\nTask 2 Configuration:")
print(f"Target gates: {TASK2_TARGET_GATES}")
print(f"Model: {TASK2_MODEL}")
print(f"Prompt length: {len(task2_prompt)} characters")

# Initialize results
task2_result = None
task2_sim_score = None
task2_evades = False
task2_functionally_correct = None

# Task 2 Execution
if OPENAI_API_KEY:
    print("\n" + "=" * 50)
    print("EXECUTING TASK 2...")
    print("=" * 50)

    try:
        response = openai.chat.completions.create(
            model=TASK2_MODEL,
            messages=[{"role": "user", "content": task2_prompt}],
            temperature=0.3,
            max_tokens=4000
        )
        llm_boolean_response = response.choices[0].message.content

        print(f"✅ LLM Response received ({len(llm_boolean_response)} chars)")
        print("First 300 chars of Boolean response:")
        print(llm_boolean_response[:300] + "...")

        # Convert back to Verilog format
        print("\n🔄 Converting back to Verilog format...")
        task2_result = convert_circuit_from_Boolean_format(llm_boolean_response, remaining_orig_lines)

        print(f"✅ Converted back to Verilog ({len(task2_result)} chars)")

        # Check functional equivalence
        print("\n🔍 Checking functional equivalence...")
        equiv_result, equiv_details = check_functional_equivalence(
            orig_file_contents, task2_result, "task2"
        )
        task2_functionally_correct = equiv_result
        print(f"Functional check: {equiv_details}")

        # Evaluate with SIM
        print("\n🔍 Evaluating with SIM detector...")
        task2_sim_score = evaluate_sim(orig_file_contents, task2_result, "task2")
        task2_evades = task2_sim_score is not None and task2_sim_score < SIM_THRESHOLD

        print(f"\n📊 TASK 2 RESULTS:")
        print(f"Functional Correctness: {'✅ YES' if task2_functionally_correct else '❌ NO' if task2_functionally_correct is False else '⚠️ UNKNOWN'}")
        print(f"SIM Score: {task2_sim_score:.4f}" if task2_sim_score else "SIM Score: Error")
        print(f"Evades Detection: {'✅ YES' if task2_evades else '❌ NO'}")
        print(f"Model Used: {TASK2_MODEL}")
        print(f"Target Gates: {TASK2_TARGET_GATES}")

        # Overall success requires BOTH functional correctness AND SIM evasion
        overall_success = task2_functionally_correct and task2_evades
        print(f"Overall Success: {'✅ YES' if overall_success else '❌ NO'}")

    except Exception as e:
        print(f"❌ Task 2 failed: {e}")
        task2_result = None
        task2_sim_score = None
        task2_evades = False
        task2_functionally_correct = None
else:
    print("⚠️ Skipping Task 2 execution (no API key)")

Boolean Format Conversion:
Boolean equations: 212 lines
Remaining structural lines: 13 lines

First 300 chars of Boolean format:
KeyWire_0[0] = NOT(N1)
N118 = XNOR(keyIn_0_0, KeyWire_0[0])
N119 = NOT(N4)
KeyWire_0[1] = NOT(N11)
N122 = XNOR(keyIn_0_1, KeyWire_0[1])
N123 = NOT(N17)
KeyWire_0[2] = NOT(N24)
KeyNOTWire_0[0] = XNOR(keyIn_0_2, KeyWire_0[2])
N126 = NOT(KeyNOTWire_0[0])
N127 = NOT(N30)
KeyWire_0[3] = NOT(N37)
N130 = X...

Task 2 Configuration:
Target gates: NOR
Model: gpt-4
Prompt length: 7207 characters

EXECUTING TASK 2...
✅ LLM Response received (8711 chars)
First 300 chars of Boolean response:
KeyWire_0[0] = NOR(N1, N1)
N118 = NOR(NOR(keyIn_0_0, KeyWire_0[0]), NOR(NOR(keyIn_0_0, KeyWire_0[0]), KeyWire_0[0]))
N119 = NOR(N4, N4)
KeyWire_0[1] = NOR(N11, N11)
N122 = NOR(NOR(keyIn_0_1, KeyWire_0[1]), NOR(NOR(keyIn_0_1, KeyWire_0[1]), KeyWire_0[1]))
N123 = NOR(N17, N17)
KeyWire_0[2] = NOR(N24, ...

🔄 Converting back to Verilog format...
✅ Converted back to Verilog (10717 chars)



In [20]:
from IPython.display import display, Markdown

analysis_md = """
## Task 2 Analysis – Boolean Format Rewriting

**Model used:** gpt-3.5-turbo-16k and gpt-4
**SIM Score:** 0.2200
**Success/Failure:** Partial success

### Success Rate
The LLM partially succeeded. Both GPT-3.5 and GPT-4 were able to rewrite the Boolean equations using NOR-only logic, and the conversion back to Verilog completed. However, the final Verilog failed during Yosys parsing due to a syntax error ("unexpected TOK_INPUT"), indicating malformed structure. Therefore, the rewrite was not fully successful.

### SIM Evasion
The SIM score was 0.2200, which is below the threshold of 0.3. Both models successfully evaded SIM-based piracy detection, showing that the rewritten circuit was structurally different from the original.

### Functional Correctness
Functional correctness could not be verified because the generated Verilog could not be compiled. The syntax error suggests issues in the reconstruction process from Boolean format back to Verilog. Normally, correctness would be verified using simulation or formal equivalence checking.

### Prompt Engineering
Compared to Task 1, the Task 2 prompt is more structured and easier for the LLM because it operates on Boolean equations instead of full Verilog. However, Task 2 is more sensitive to formatting errors. Even small deviations in output format can break the conversion pipeline. Therefore, strict formatting constraints are critical.

### Model Comparison
Both GPT-3.5 and GPT-4 produced similar results. GPT-4 generated slightly cleaner Boolean expressions, but both models failed at the final Verilog conversion stage. This suggests that the bottleneck is not only model capability, but also the strict parsing requirements.

### Key Insights
Boolean format simplifies the rewriting task and improves LLM performance compared to direct Verilog rewriting. However, the pipeline is fragile, and small formatting errors can cause failure in later stages. SIM evasion is relatively easy to achieve, but ensuring functional correctness remains challenging.
"""

display(Markdown(analysis_md))


## Task 2 Analysis – Boolean Format Rewriting

**Model used:** gpt-3.5-turbo-16k and gpt-4  
**SIM Score:** 0.2200  
**Success/Failure:** Partial success  

### Success Rate
The LLM partially succeeded. Both GPT-3.5 and GPT-4 were able to rewrite the Boolean equations using NOR-only logic, and the conversion back to Verilog completed. However, the final Verilog failed during Yosys parsing due to a syntax error ("unexpected TOK_INPUT"), indicating malformed structure. Therefore, the rewrite was not fully successful.

### SIM Evasion
The SIM score was 0.2200, which is below the threshold of 0.3. Both models successfully evaded SIM-based piracy detection, showing that the rewritten circuit was structurally different from the original.

### Functional Correctness
Functional correctness could not be verified because the generated Verilog could not be compiled. The syntax error suggests issues in the reconstruction process from Boolean format back to Verilog. Normally, correctness would be verified using simulation or formal equivalence checking.

### Prompt Engineering
Compared to Task 1, the Task 2 prompt is more structured and easier for the LLM because it operates on Boolean equations instead of full Verilog. However, Task 2 is more sensitive to formatting errors. Even small deviations in output format can break the conversion pipeline. Therefore, strict formatting constraints are critical.

### Model Comparison
Both GPT-3.5 and GPT-4 produced similar results. GPT-4 generated slightly cleaner Boolean expressions, but both models failed at the final Verilog conversion stage. This suggests that the bottleneck is not only model capability, but also the strict parsing requirements.

### Key Insights
Boolean format simplifies the rewriting task and improves LLM performance compared to direct Verilog rewriting. However, the pipeline is fragile, and small formatting errors can cause failure in later stages. SIM evasion is relatively easy to achieve, but ensuring functional correctness remains challenging.


In [23]:



#@title Task 3: Divide & Conquer (No Feedback) Implementation

# Characterize the circuit to get all gate types and query templates
# (This is the "divide" step: one LLM query per gate type)
gate_counts = characterize_generic_netlist(orig_file_contents)
query_templates = create_LLM_circuit_query_template(gate_counts)

# Gates skipped by design: trivial (NOT/BUF) need no rephrasing
SKIP_GATE_TYPES = ['not_1', 'buf_1']
non_trivial_gates = [g for g in gate_counts if g not in SKIP_GATE_TYPES]
skipped_gates    = [g for g in gate_counts if g in SKIP_GATE_TYPES]

print("Gate types found:", list(gate_counts.keys()))
print(f"Non-trivial gate types to map: {non_trivial_gates}")
if skipped_gates:
    print(f"Skipped (trivial, no rephrasing needed): {skipped_gates}")

# TODO: Students configure task 3
TASK3_TARGET_GATES = ["NOR"]  # Try: ["NOR"], ["NAND"], ["OR", "NOT"]
TASK3_MODEL = "gpt-3.5-turbo-16k"

# TODO: Students should improve this prompt (from original GPT_communication_script.py)

def create_task3_prompt(template, target_gates):
    if len(target_gates) == 1:
        gate_name = target_gates[0]
        gate_rule = f"Use ONLY {gate_name} gates."
        gate_format = f"{gate_name}(a, b)"
    else:
        gate_rule = f"Use ONLY {' and '.join(target_gates)} gates."
        gate_format = f"{target_gates[0]}(a, b)"

    return f"""
Rewrite the following logic gate using ONLY the allowed gate set.

Strict rules:
1. {gate_rule}
2. Preserve EXACT same functionality.
3. Output EXACTLY ONE line.
4. Format MUST be:
   output = {gate_format}
5. Do NOT include explanation, comments, or extra text.
6. Do NOT include multiple lines.
7. Do NOT rename variables.
8. Do NOT skip inputs.

Important:
- NOT(x) must be implemented using {target_gates[0]}(x, x)
- Every expression must be a valid gate call
- No plain Boolean expressions (like a & b)

---

Input gate:
{template}

---

Return ONLY the rewritten gate in the required format.
"""

print(f"\nTask 3 Configuration:")
print(f"Target gates: {TASK3_TARGET_GATES}")
print(f"Model: {TASK3_MODEL}")

# Initialize results
task3_circuit_mapping = {}
task3_final_circuit = None
task3_sim_score = None
task3_evades = False
task3_functionally_correct = None

# Task 3 Execution
if OPENAI_API_KEY:
    print("\n" + "=" * 50)
    print("EXECUTING TASK 3...")
    print("=" * 50)

    try:
        target_key = "_".join(TASK3_TARGET_GATES)

        # Initialize all gate types with empty mappings (unmapped gates stay as-is)
        for gate_type in gate_counts:
            task3_circuit_mapping[gate_type] = {}

        # For each non-trivial gate type, ask the LLM for a single-shot rephrasing
        for gate_type, template in query_templates.items():
            if gate_type in SKIP_GATE_TYPES:
                continue

            print(f"\n--- Gate type: {gate_type}  (template: {template}) ---")
            prompt = create_task3_prompt(template, TASK3_TARGET_GATES)

            response = openai.chat.completions.create(
                model=TASK3_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,
                max_tokens=500
            )

            llm_response = response.choices[0].message.content
            LLM_circuit = get_circuit_from_response(llm_response)

            if "Incorrect format" in LLM_circuit:
                print(f"  ⚠️ Could not parse LLM response — keeping original gate")
                continue

            print(f"  ✅ Mapping: {LLM_circuit.strip()}")
            task3_circuit_mapping[gate_type][target_key] = [LLM_circuit]

        mapped_count = sum(1 for g in non_trivial_gates if task3_circuit_mapping.get(g))
        print(f"\n✅ Got mappings for {mapped_count}/{len(non_trivial_gates)} non-trivial gate types")
        if skipped_gates:
            print(f"   (Skipped by design: {skipped_gates})")

        # Apply all mappings to the full circuit ("conquer" step)
        print("\n🔄 Applying gate mappings to full circuit...")
        task3_final_circuit = get_mapped_circuit(orig_file_contents, task3_circuit_mapping, target_key)

        # Check functional equivalence
        print("\n🔍 Checking functional equivalence...")
        equiv_result, equiv_details = check_functional_equivalence(
            orig_file_contents, task3_final_circuit, "task3"
        )
        task3_functionally_correct = equiv_result
        print(f"Functional check: {equiv_details}")

        # Evaluate with SIM
        print("\n🔍 Evaluating with SIM detector...")
        task3_sim_score = evaluate_sim(orig_file_contents, task3_final_circuit, "task3")
        task3_evades = task3_sim_score is not None and task3_sim_score < SIM_THRESHOLD

        print(f"\n📊 TASK 3 RESULTS:")
        print(f"Gate types mapped: {mapped_count}/{len(non_trivial_gates)} non-trivial  (skipped: {skipped_gates})")
        print(f"Functional Correctness: {'✅ YES' if task3_functionally_correct else '❌ NO' if task3_functionally_correct is False else '⚠️ UNKNOWN'}")
        print(f"SIM Score: {task3_sim_score:.4f}" if task3_sim_score is not None else "SIM Score: Error")
        print(f"Evades Detection: {'✅ YES' if task3_evades else '❌ NO'}")
        print(f"Model Used: {TASK3_MODEL}")
        print(f"Target Gates: {TASK3_TARGET_GATES}")
        overall_success = task3_functionally_correct and task3_evades
        print(f"Overall Success: {'✅ YES' if overall_success else '❌ NO'}")

    except Exception as e:
        print(f"❌ Task 3 failed: {e}")
        task3_circuit_mapping = {}
        task3_final_circuit = None
        task3_sim_score = None
        task3_evades = False
        task3_functionally_correct = None
else:
    print("⚠️ Skipping Task 3 execution (no API key)")


Gate types found: ['nand_2', 'not_1', 'xor_2', 'nor_2', 'xnor_2', 'nand_4', 'and_9', 'and_8', 'nand_3']
Non-trivial gate types to map: ['nand_2', 'xor_2', 'nor_2', 'xnor_2', 'nand_4', 'and_9', 'and_8', 'nand_3']
Skipped (trivial, no rephrasing needed): ['not_1']

Task 3 Configuration:
Target gates: ['NOR']
Model: gpt-3.5-turbo-16k

EXECUTING TASK 3...

--- Gate type: nand_2  (template: Y = NAND(A1, A2)) ---
  ✅ Mapping: my_N189 = NOR(A1, A1)
my_N190 = NOR(A2, A2)
output = NOR(my_N189, my_N190)

--- Gate type: xor_2  (template: Y = XOR(A1, A2)) ---
  ✅ Mapping: my_N192 = NOR(A1, A1)
my_N193 = NOR(A2, A2)
my_N194 = NOR(my_N192, my_N193)
my_N195 = NOR(A1, A2)
my_N196 = NOR(A2, A1)
my_N197 = NOR(my_N195, my_N196)
output = NOR(my_N194, my_N197)

--- Gate type: nor_2  (template: Y = NOR(A1, A2)) ---
  ✅ Mapping: my_N199 = NOR(A1, A2)
my_N200 = NOR(A1, A2)
output = NOR(my_N199, my_N200)

--- Gate type: xnor_2  (template: Y = XNOR(A1, A2)) ---
  ✅ Mapping: my_N202 = NOR(A1, A2)
my_N203 = NOR(A

In [24]:



#@title Task 3: Divide & Conquer (No Feedback) Implementation

# Characterize the circuit to get all gate types and query templates
# (This is the "divide" step: one LLM query per gate type)
gate_counts = characterize_generic_netlist(orig_file_contents)
query_templates = create_LLM_circuit_query_template(gate_counts)

# Gates skipped by design: trivial (NOT/BUF) need no rephrasing
SKIP_GATE_TYPES = ['not_1', 'buf_1']
non_trivial_gates = [g for g in gate_counts if g not in SKIP_GATE_TYPES]
skipped_gates    = [g for g in gate_counts if g in SKIP_GATE_TYPES]

print("Gate types found:", list(gate_counts.keys()))
print(f"Non-trivial gate types to map: {non_trivial_gates}")
if skipped_gates:
    print(f"Skipped (trivial, no rephrasing needed): {skipped_gates}")

# TODO: Students configure task 3
TASK3_TARGET_GATES = ["NOR"]  # Try: ["NOR"], ["NAND"], ["OR", "NOT"]
TASK3_MODEL = "gpt-4"

# TODO: Students should improve this prompt (from original GPT_communication_script.py)

def create_task3_prompt(template, target_gates):
    if len(target_gates) == 1:
        gate_name = target_gates[0]
        gate_rule = f"Use ONLY {gate_name} gates."
        gate_format = f"{gate_name}(a, b)"
    else:
        gate_rule = f"Use ONLY {' and '.join(target_gates)} gates."
        gate_format = f"{target_gates[0]}(a, b)"

    return f"""
Rewrite the following logic gate using ONLY the allowed gate set.

Strict rules:
1. {gate_rule}
2. Preserve EXACT same functionality.
3. Output EXACTLY ONE line.
4. Format MUST be:
   output = {gate_format}
5. Do NOT include explanation, comments, or extra text.
6. Do NOT include multiple lines.
7. Do NOT rename variables.
8. Do NOT skip inputs.

Important:
- NOT(x) must be implemented using {target_gates[0]}(x, x)
- Every expression must be a valid gate call
- No plain Boolean expressions (like a & b)

---

Input gate:
{template}

---

Return ONLY the rewritten gate in the required format.
"""

print(f"\nTask 3 Configuration:")
print(f"Target gates: {TASK3_TARGET_GATES}")
print(f"Model: {TASK3_MODEL}")

# Initialize results
task3_circuit_mapping = {}
task3_final_circuit = None
task3_sim_score = None
task3_evades = False
task3_functionally_correct = None

# Task 3 Execution
if OPENAI_API_KEY:
    print("\n" + "=" * 50)
    print("EXECUTING TASK 3...")
    print("=" * 50)

    try:
        target_key = "_".join(TASK3_TARGET_GATES)

        # Initialize all gate types with empty mappings (unmapped gates stay as-is)
        for gate_type in gate_counts:
            task3_circuit_mapping[gate_type] = {}

        # For each non-trivial gate type, ask the LLM for a single-shot rephrasing
        for gate_type, template in query_templates.items():
            if gate_type in SKIP_GATE_TYPES:
                continue

            print(f"\n--- Gate type: {gate_type}  (template: {template}) ---")
            prompt = create_task3_prompt(template, TASK3_TARGET_GATES)

            response = openai.chat.completions.create(
                model=TASK3_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,
                max_tokens=500
            )

            llm_response = response.choices[0].message.content
            LLM_circuit = get_circuit_from_response(llm_response)

            if "Incorrect format" in LLM_circuit:
                print(f"  ⚠️ Could not parse LLM response — keeping original gate")
                continue

            print(f"  ✅ Mapping: {LLM_circuit.strip()}")
            task3_circuit_mapping[gate_type][target_key] = [LLM_circuit]

        mapped_count = sum(1 for g in non_trivial_gates if task3_circuit_mapping.get(g))
        print(f"\n✅ Got mappings for {mapped_count}/{len(non_trivial_gates)} non-trivial gate types")
        if skipped_gates:
            print(f"   (Skipped by design: {skipped_gates})")

        # Apply all mappings to the full circuit ("conquer" step)
        print("\n🔄 Applying gate mappings to full circuit...")
        task3_final_circuit = get_mapped_circuit(orig_file_contents, task3_circuit_mapping, target_key)

        # Check functional equivalence
        print("\n🔍 Checking functional equivalence...")
        equiv_result, equiv_details = check_functional_equivalence(
            orig_file_contents, task3_final_circuit, "task3"
        )
        task3_functionally_correct = equiv_result
        print(f"Functional check: {equiv_details}")

        # Evaluate with SIM
        print("\n🔍 Evaluating with SIM detector...")
        task3_sim_score = evaluate_sim(orig_file_contents, task3_final_circuit, "task3")
        task3_evades = task3_sim_score is not None and task3_sim_score < SIM_THRESHOLD

        print(f"\n📊 TASK 3 RESULTS:")
        print(f"Gate types mapped: {mapped_count}/{len(non_trivial_gates)} non-trivial  (skipped: {skipped_gates})")
        print(f"Functional Correctness: {'✅ YES' if task3_functionally_correct else '❌ NO' if task3_functionally_correct is False else '⚠️ UNKNOWN'}")
        print(f"SIM Score: {task3_sim_score:.4f}" if task3_sim_score is not None else "SIM Score: Error")
        print(f"Evades Detection: {'✅ YES' if task3_evades else '❌ NO'}")
        print(f"Model Used: {TASK3_MODEL}")
        print(f"Target Gates: {TASK3_TARGET_GATES}")
        overall_success = task3_functionally_correct and task3_evades
        print(f"Overall Success: {'✅ YES' if overall_success else '❌ NO'}")

    except Exception as e:
        print(f"❌ Task 3 failed: {e}")
        task3_circuit_mapping = {}
        task3_final_circuit = None
        task3_sim_score = None
        task3_evades = False
        task3_functionally_correct = None
else:
    print("⚠️ Skipping Task 3 execution (no API key)")


Gate types found: ['nand_2', 'not_1', 'xor_2', 'nor_2', 'xnor_2', 'nand_4', 'and_9', 'and_8', 'nand_3']
Non-trivial gate types to map: ['nand_2', 'xor_2', 'nor_2', 'xnor_2', 'nand_4', 'and_9', 'and_8', 'nand_3']
Skipped (trivial, no rephrasing needed): ['not_1']

Task 3 Configuration:
Target gates: ['NOR']
Model: gpt-4

EXECUTING TASK 3...

--- Gate type: nand_2  (template: Y = NAND(A1, A2)) ---
  ✅ Mapping: my_N286 = NOR(A1, A1)
my_N287 = NOR(A2, A2)
Y = NOR(my_N286, my_N287)

--- Gate type: xor_2  (template: Y = XOR(A1, A2)) ---
  ✅ Mapping: my_N289 = NOR(A1, A1)
my_N290 = NOR(A2, A2)
my_N291 = NOR(my_N289, my_N290)
my_N292 = NOR(A1, A2)
my_N293 = NOR(A1, A2)
my_N294 = NOR(my_N292, my_N293)
Y = NOR(my_N291, my_N294)

--- Gate type: nor_2  (template: Y = NOR(A1, A2)) ---
  ✅ Mapping: Y = NOR(A1, A2)

--- Gate type: xnor_2  (template: Y = XNOR(A1, A2)) ---
  ✅ Mapping: my_N297 = NOR(A1, A2)
my_N298 = NOR(A1, A1)
my_N299 = NOR(A2, A2)
my_N300 = NOR(my_N298, my_N299)
Y = NOR(my_N297, my_

In [25]:
from IPython.display import display, Markdown

analysis_md = """
## Task 3 Analysis – Divide & Conquer Mapping (No Feedback)

**Model used:** gpt-4
**Target gates:** NOR
**Gate types mapped:** 8/8 non-trivial gate types
**SIM Score:** 0.3100
**Success/Failure:** Failure

### Success Rate
The LLM successfully generated mappings for all 8 non-trivial gate types, including `nand_2`, `xor_2`, `nor_2`, `xnor_2`, `nand_4`, `and_9`, `and_8`, and `nand_3`. The trivial gate type `not_1` was skipped by design. This shows that the divide-and-conquer strategy made the generation step much easier and more efficient, since each query focused on a single gate pattern instead of the entire circuit.

However, the final mapped circuit was not successful overall. Although every gate type received a mapping, the reconstructed full circuit was not functionally equivalent to the original design.

### SIM Evasion
The SIM score was **0.3100**, which is slightly above the evasion threshold of **0.3**. Therefore, the rewritten circuit did **not** evade SIM-based piracy detection.

Compared to earlier tasks, the SIM score became higher rather than lower. This suggests that divide-and-conquer mapping may preserve too much of the original circuit structure, even if the local gate implementations are rewritten. In other words, the gate-level replacements were not sufficient to make the full design look different enough to the detector.

### Functional Correctness
The functional equivalence check failed. Yosys reported that the circuits were **not equivalent** and found a counterexample using SAT. This means the rewritten circuit compiles, but does not preserve the same logic behavior as the original circuit.

A likely reason is that some of the generated gate mappings were logically incorrect, especially for larger multi-input gates such as `and_8` and `and_9`. For example, the generated mappings used a single multi-input NOR expression at the output, which may not correctly implement the intended AND behavior under the expected gate semantics. Since Task 3 uses no feedback loop, even one incorrect gate mapping propagates into the final circuit and breaks equivalence.

### Prompt Engineering
Compared to Task 1 and Task 2, the Task 3 prompt is much shorter and more focused. Instead of rewriting a full circuit or a large set of Boolean equations, the model only needs to rewrite one gate template at a time. This reduces task complexity and improves response speed.

However, because there is no feedback or verification during the per-gate generation stage, the prompt must be extremely precise. A mapping may look syntactically reasonable but still be logically wrong. This shows that for divide-and-conquer approaches, output format is easier to control, but logical correctness of each reusable mapping becomes the main challenge.

### Model Comparison
In this run, GPT-4 successfully generated mappings for all gate types very quickly, which suggests stronger local reasoning and cleaner formatting compared to weaker models. However, faster generation and complete coverage did not guarantee correctness. The final circuit still failed equivalence checking and did not evade SIM detection.

This indicates that GPT-4 improves the mapping generation process, but without a retry or correction mechanism, incorrect mappings can still make the entire strategy fail.

### Key Insights
The divide-and-conquer approach improves efficiency because each LLM query is smaller, more focused, and easier to parse. In this experiment, all gate mappings were generated successfully, and the generation process was faster than previous tasks.

However, this method did not improve the final outcome. The functional equivalence check failed, and the SIM score was slightly worse than before. This suggests that divide-and-conquer without feedback is efficient but not robust. It can generate reusable gate mappings quickly, but if even one mapping is logically wrong, the entire circuit fails. In addition, replacing gates locally may not change the overall circuit structure enough to evade similarity-based detectors.
"""

display(Markdown(analysis_md))


## Task 3 Analysis – Divide & Conquer Mapping (No Feedback)

**Model used:** gpt-4  
**Target gates:** NOR  
**Gate types mapped:** 8/8 non-trivial gate types  
**SIM Score:** 0.3100  
**Success/Failure:** Failure  

### Success Rate
The LLM successfully generated mappings for all 8 non-trivial gate types, including `nand_2`, `xor_2`, `nor_2`, `xnor_2`, `nand_4`, `and_9`, `and_8`, and `nand_3`. The trivial gate type `not_1` was skipped by design. This shows that the divide-and-conquer strategy made the generation step much easier and more efficient, since each query focused on a single gate pattern instead of the entire circuit.

However, the final mapped circuit was not successful overall. Although every gate type received a mapping, the reconstructed full circuit was not functionally equivalent to the original design.

### SIM Evasion
The SIM score was **0.3100**, which is slightly above the evasion threshold of **0.3**. Therefore, the rewritten circuit did **not** evade SIM-based piracy detection.

Compared to earlier tasks, the SIM score became higher rather than lower. This suggests that divide-and-conquer mapping may preserve too much of the original circuit structure, even if the local gate implementations are rewritten. In other words, the gate-level replacements were not sufficient to make the full design look different enough to the detector.

### Functional Correctness
The functional equivalence check failed. Yosys reported that the circuits were **not equivalent** and found a counterexample using SAT. This means the rewritten circuit compiles, but does not preserve the same logic behavior as the original circuit.

A likely reason is that some of the generated gate mappings were logically incorrect, especially for larger multi-input gates such as `and_8` and `and_9`. For example, the generated mappings used a single multi-input NOR expression at the output, which may not correctly implement the intended AND behavior under the expected gate semantics. Since Task 3 uses no feedback loop, even one incorrect gate mapping propagates into the final circuit and breaks equivalence.

### Prompt Engineering
Compared to Task 1 and Task 2, the Task 3 prompt is much shorter and more focused. Instead of rewriting a full circuit or a large set of Boolean equations, the model only needs to rewrite one gate template at a time. This reduces task complexity and improves response speed.

However, because there is no feedback or verification during the per-gate generation stage, the prompt must be extremely precise. A mapping may look syntactically reasonable but still be logically wrong. This shows that for divide-and-conquer approaches, output format is easier to control, but logical correctness of each reusable mapping becomes the main challenge.

### Model Comparison
In this run, GPT-4 successfully generated mappings for all gate types very quickly, which suggests stronger local reasoning and cleaner formatting compared to weaker models. However, faster generation and complete coverage did not guarantee correctness. The final circuit still failed equivalence checking and did not evade SIM detection.

This indicates that GPT-4 improves the mapping generation process, but without a retry or correction mechanism, incorrect mappings can still make the entire strategy fail.

### Key Insights
The divide-and-conquer approach improves efficiency because each LLM query is smaller, more focused, and easier to parse. In this experiment, all gate mappings were generated successfully, and the generation process was faster than previous tasks.

However, this method did not improve the final outcome. The functional equivalence check failed, and the SIM score was slightly worse than before. This suggests that divide-and-conquer without feedback is efficient but not robust. It can generate reusable gate mappings quickly, but if even one mapping is logically wrong, the entire circuit fails. In addition, replacing gates locally may not change the overall circuit structure enough to evade similarity-based detectors.


In [26]:
#@title Task 4: Divide & Conquer WITH Iterative Feedback Implementation

# Reuse gate characterization from Task 3
# (gate_counts, query_templates, SKIP_GATE_TYPES,
#  non_trivial_gates, skipped_gates are already set)
print(f"Gate types to process: {non_trivial_gates}")
if skipped_gates:
    print(f"Skipped (trivial): {skipped_gates}")

# TODO: Students configure task 4
TASK4_TARGET_GATES = ["NOR"]  # Try: ["NOR"], ["NAND"], ["OR", "NOT"]
TASK4_MODEL = "gpt-4"  # Try: "gpt-4" for better reasoning
TASK4_MAX_TRIALS = 5  # Max feedback rounds per gate type (matches original source)

# TODO: Students should improve this prompt (from original GPT_communication_script.py)

def create_task4_prompt(template, target_gates):
    if len(target_gates) == 1:
        gate_name = target_gates[0]
        gate_rule = f"Use ONLY {gate_name} gates."
        gate_call = f"{gate_name}(A1, A2)"
    else:
        gate_rule = f"Use ONLY {' and '.join(target_gates)} gates."
        gate_call = f"{target_gates[0]}(A1, A2)"

    return f"""
Rewrite the following gate implementation using ONLY the allowed gate set.

Strict requirements:
1. {gate_rule}
2. Preserve EXACT same functionality.
3. Return ONLY the rewritten circuit.
4. Do NOT include any explanation, comments, markdown, numbering, or extra text.
5. Keep the same variable names for inputs and output.
6. You may introduce intermediate wires if needed.
7. Every assignment must be in the form:
   X = GATE(arg1, arg2)
8. Return a valid structural decomposition only.
9. Do NOT use plain Boolean operators such as &, |, ~, ^.
10. If inversion is needed, implement it structurally using {target_gates[0]}(x, x).

Important:
- Multi-line output is allowed if intermediate signals are needed.
- The final output variable must be Y.
- Every right-hand side must be a valid gate call.
- Do NOT skip any input.

Original gate template:
{template}

Return ONLY the rewritten circuit.
"""

def create_task4_format_feedback(template, target_gates):
    if len(target_gates) == 1:
        gate_name = target_gates[0]
    else:
        gate_name = target_gates[0]

    return f"""
Your previous answer was rejected because the format was incorrect.

Try again and follow these rules exactly:
1. Return ONLY the rewritten circuit.
2. Do NOT include explanation, comments, markdown, or extra text.
3. Use ONLY {', '.join(target_gates)} gate(s).
4. Each line must be exactly in this form:
   X = {gate_name}(arg1, arg2)
5. Keep the same input names and output name Y.
6. If inversion is needed, use {gate_name}(x, x).
7. Every right-hand side must be a valid gate call.

Original gate template:
{template}

Return ONLY the corrected rewritten circuit.
"""

def create_task4_functional_feedback(template, target_gates):
    return f"""
Your previous answer was rejected because it did not preserve the same functionality as the original gate.

Try again and follow these rules exactly:
1. Preserve EXACT logical functionality.
2. Use ONLY {', '.join(target_gates)} gate(s).
3. Return ONLY the rewritten circuit.
4. Do NOT include explanation, comments, markdown, or extra text.
5. Keep the same input names and output name Y.
6. You may introduce intermediate signals if needed.
7. Every line must be in the form:
   X = {target_gates[0]}(arg1, arg2)

Original gate template:
{template}

Return ONLY a functionally correct rewritten circuit.
"""


def create_task4_format_feedback(template, target_gates):
    return (
        "This is not the correct format. Can you try again in this format: "
        " =  ()? "
        f"Use only {', '.join(target_gates)} operator(s). "
        "Below is the original circuit:\n" + template
    )

def create_task4_functional_feedback(template, target_gates):
    return (
        "This is not correct because the functionality is not the same as the original circuit. "
        "Can you try again? Below is the original circuit:\n" + template
    )

print(f"\nTask 4 Configuration:")
print(f"Target gates: {TASK4_TARGET_GATES}")
print(f"Model: {TASK4_MODEL}")
print(f"Max trials per gate type: {TASK4_MAX_TRIALS}")

# Initialize results
task4_circuit_mapping = {}
task4_final_circuit = None
task4_sim_score = None
task4_evades = False
task4_functionally_correct = None
task4_gate_trial_counts = {}

# Task 4 Execution
if OPENAI_API_KEY:
    print("\n" + "=" * 50)
    print("EXECUTING TASK 4...")
    print("=" * 50)

    try:
        target_key = "_".join(TASK4_TARGET_GATES)

        # Initialize all gate types with empty mappings
        for gate_type in gate_counts:
            task4_circuit_mapping[gate_type] = {}

        # For each non-trivial gate type, query LLM with iterative feedback on failure
        for gate_type, template in query_templates.items():
            if gate_type in SKIP_GATE_TYPES:
                continue

            print(f"\n--- Gate type: {gate_type}  (template: {template}) ---")
            messages = [{"role": "user", "content": create_task4_prompt(template, TASK4_TARGET_GATES)}]
            num_trials = 0
            success = False

            while num_trials < TASK4_MAX_TRIALS:
                num_trials += 1
                print(f"  Trial {num_trials}/{TASK4_MAX_TRIALS}")

                # Use full conversation history for context (feedback loop)
                response = openai.chat.completions.create(
                    model=TASK4_MODEL,
                    messages=messages,
                    temperature=0.2,
                    max_tokens=500
                )
                llm_response = response.choices[0].message.content
                messages.append({"role": "assistant", "content": llm_response})

                LLM_circuit = get_circuit_from_response(llm_response)

                if "Incorrect format" in LLM_circuit:
                    print(f"  ⚠️ Format error — sending feedback")
                    feedback = create_task4_format_feedback(template, TASK4_TARGET_GATES)
                    messages.append({"role": "user", "content": feedback})
                    continue

                print(f"  ✅ Mapping: {LLM_circuit.strip()}")
                task4_circuit_mapping[gate_type][target_key] = [LLM_circuit]
                success = True
                break

            task4_gate_trial_counts[gate_type] = num_trials
            if not success:
                print(f"  ❌ No valid mapping after {TASK4_MAX_TRIALS} trials — keeping original gate")

        mapped_count = sum(1 for g in non_trivial_gates if task4_circuit_mapping.get(g))
        avg_trials = sum(task4_gate_trial_counts.values()) / len(task4_gate_trial_counts) if task4_gate_trial_counts else 0
        print(f"\n✅ Got mappings for {mapped_count}/{len(non_trivial_gates)} non-trivial gate types")
        if skipped_gates:
            print(f"   (Skipped by design: {skipped_gates})")
        print(f"Average trials per gate type: {avg_trials:.1f}")

        # Apply all mappings to the full circuit
        print("\n🔄 Applying gate mappings to full circuit...")
        task4_final_circuit = get_mapped_circuit(orig_file_contents, task4_circuit_mapping, target_key)

        # Check functional equivalence on the full mapped circuit
        print("\n🔍 Checking functional equivalence...")
        equiv_result, equiv_details = check_functional_equivalence(
            orig_file_contents, task4_final_circuit, "task4"
        )
        task4_functionally_correct = equiv_result
        print(f"Functional check: {equiv_details}")

        # Evaluate with SIM
        print("\n🔍 Evaluating with SIM detector...")
        task4_sim_score = evaluate_sim(orig_file_contents, task4_final_circuit, "task4")
        task4_evades = task4_sim_score is not None and task4_sim_score < SIM_THRESHOLD

        print(f"\n📊 TASK 4 RESULTS:")
        print(f"Gate types mapped: {mapped_count}/{len(non_trivial_gates)} non-trivial  (skipped: {skipped_gates})")
        print(f"Average trials per gate type: {avg_trials:.1f}")
        print(f"Functional Correctness: {'✅ YES' if task4_functionally_correct else '❌ NO' if task4_functionally_correct is False else '⚠️ UNKNOWN'}")
        print(f"SIM Score: {task4_sim_score:.4f}" if task4_sim_score is not None else "SIM Score: Error")
        print(f"Evades Detection: {'✅ YES' if task4_evades else '❌ NO'}")
        print(f"Model Used: {TASK4_MODEL}")
        print(f"Target Gates: {TASK4_TARGET_GATES}")
        overall_success = task4_functionally_correct and task4_evades
        print(f"Overall Success: {'✅ YES' if overall_success else '❌ NO'}")

        # Show per-gate-type trial counts
        print("\nPer-gate-type trial counts:")
        for gt in gate_counts:
            if gt in SKIP_GATE_TYPES:
                print(f"  {gt}: — (trivial gate, skipped by design)")
            elif gt in task4_gate_trial_counts:
                trials = task4_gate_trial_counts[gt]
                mapped = '✅' if task4_circuit_mapping.get(gt) else '❌'
                print(f"  {gt}: {trials} trial(s) {mapped}")

    except Exception as e:
        print(f"❌ Task 4 failed: {e}")
        task4_circuit_mapping = {}
        task4_final_circuit = None
        task4_sim_score = None
        task4_evades = False
        task4_functionally_correct = None
        task4_gate_trial_counts = {}
else:
    print("⚠️ Skipping Task 4 execution (no API key)")



Gate types to process: ['nand_2', 'xor_2', 'nor_2', 'xnor_2', 'nand_4', 'and_9', 'and_8', 'nand_3']
Skipped (trivial): ['not_1']

Task 4 Configuration:
Target gates: ['NOR']
Model: gpt-4
Max trials per gate type: 5

EXECUTING TASK 4...

--- Gate type: nand_2  (template: Y = NAND(A1, A2)) ---
  Trial 1/5
  ✅ Mapping: N1 = NOR(A1, A1)
N2 = NOR(A2, A2)
Y = NOR(N1, N2)

--- Gate type: xor_2  (template: Y = XOR(A1, A2)) ---
  Trial 1/5
  ✅ Mapping: N1 = NOR(A1, A1)
N2 = NOR(A2, A2)
N3 = NOR(A1, A2)
N4 = NOR(N1, N2)
Y = NOR(N3, N4)

--- Gate type: nor_2  (template: Y = NOR(A1, A2)) ---
  Trial 1/5
  ✅ Mapping: Y = NOR(A1, A2)

--- Gate type: xnor_2  (template: Y = XNOR(A1, A2)) ---
  Trial 1/5
  ✅ Mapping: N1 = NOR(A1, A1)
N2 = NOR(A2, A2)
N3 = NOR(A1, A2)
N4 = NOR(N1, N2)
Y = NOR(N3, N4)

--- Gate type: nand_4  (template: Y = NAND(A1, A2, A3, A4)) ---
  Trial 1/5
  ✅ Mapping: N1 = NOR(A1, A1)
N2 = NOR(A2, A2)
N3 = NOR(A3, A3)
N4 = NOR(A4, A4)
N5 = NOR(N1, N2)
N6 = NOR(N3, N4)
Y = NOR(N5, N6

In [27]:
#@title Task 4: Divide & Conquer WITH Iterative Feedback Implementation

# Reuse gate characterization from Task 3
# (gate_counts, query_templates, SKIP_GATE_TYPES,
#  non_trivial_gates, skipped_gates are already set)
print(f"Gate types to process: {non_trivial_gates}")
if skipped_gates:
    print(f"Skipped (trivial): {skipped_gates}")

# TODO: Students configure task 4
TASK4_TARGET_GATES = ["NAND"]  # Try: ["NOR"], ["NAND"], ["OR", "NOT"]
TASK4_MODEL = "gpt-4"  # Try: "gpt-4" for better reasoning
TASK4_MAX_TRIALS = 5  # Max feedback rounds per gate type (matches original source)

# TODO: Students should improve this prompt (from original GPT_communication_script.py)

def create_task4_prompt(template, target_gates):
    if len(target_gates) == 1:
        gate_name = target_gates[0]
        gate_rule = f"Use ONLY {gate_name} gates."
        gate_call = f"{gate_name}(A1, A2)"
    else:
        gate_rule = f"Use ONLY {' and '.join(target_gates)} gates."
        gate_call = f"{target_gates[0]}(A1, A2)"

    return f"""
Rewrite the following gate implementation using ONLY the allowed gate set.

Strict requirements:
1. {gate_rule}
2. Preserve EXACT same functionality.
3. Return ONLY the rewritten circuit.
4. Do NOT include any explanation, comments, markdown, numbering, or extra text.
5. Keep the same variable names for inputs and output.
6. You may introduce intermediate wires if needed.
7. Every assignment must be in the form:
   X = GATE(arg1, arg2)
8. Return a valid structural decomposition only.
9. Do NOT use plain Boolean operators such as &, |, ~, ^.
10. If inversion is needed, implement it structurally using {target_gates[0]}(x, x).

Important:
- Multi-line output is allowed if intermediate signals are needed.
- The final output variable must be Y.
- Every right-hand side must be a valid gate call.
- Do NOT skip any input.

Original gate template:
{template}

Return ONLY the rewritten circuit.
"""

def create_task4_format_feedback(template, target_gates):
    if len(target_gates) == 1:
        gate_name = target_gates[0]
    else:
        gate_name = target_gates[0]

    return f"""
Your previous answer was rejected because the format was incorrect.

Try again and follow these rules exactly:
1. Return ONLY the rewritten circuit.
2. Do NOT include explanation, comments, markdown, or extra text.
3. Use ONLY {', '.join(target_gates)} gate(s).
4. Each line must be exactly in this form:
   X = {gate_name}(arg1, arg2)
5. Keep the same input names and output name Y.
6. If inversion is needed, use {gate_name}(x, x).
7. Every right-hand side must be a valid gate call.

Original gate template:
{template}

Return ONLY the corrected rewritten circuit.
"""

def create_task4_functional_feedback(template, target_gates):
    return f"""
Your previous answer was rejected because it did not preserve the same functionality as the original gate.

Try again and follow these rules exactly:
1. Preserve EXACT logical functionality.
2. Use ONLY {', '.join(target_gates)} gate(s).
3. Return ONLY the rewritten circuit.
4. Do NOT include explanation, comments, markdown, or extra text.
5. Keep the same input names and output name Y.
6. You may introduce intermediate signals if needed.
7. Every line must be in the form:
   X = {target_gates[0]}(arg1, arg2)

Original gate template:
{template}

Return ONLY a functionally correct rewritten circuit.
"""


def create_task4_format_feedback(template, target_gates):
    return (
        "This is not the correct format. Can you try again in this format: "
        " =  ()? "
        f"Use only {', '.join(target_gates)} operator(s). "
        "Below is the original circuit:\n" + template
    )

def create_task4_functional_feedback(template, target_gates):
    return (
        "This is not correct because the functionality is not the same as the original circuit. "
        "Can you try again? Below is the original circuit:\n" + template
    )

print(f"\nTask 4 Configuration:")
print(f"Target gates: {TASK4_TARGET_GATES}")
print(f"Model: {TASK4_MODEL}")
print(f"Max trials per gate type: {TASK4_MAX_TRIALS}")

# Initialize results
task4_circuit_mapping = {}
task4_final_circuit = None
task4_sim_score = None
task4_evades = False
task4_functionally_correct = None
task4_gate_trial_counts = {}

# Task 4 Execution
if OPENAI_API_KEY:
    print("\n" + "=" * 50)
    print("EXECUTING TASK 4...")
    print("=" * 50)

    try:
        target_key = "_".join(TASK4_TARGET_GATES)

        # Initialize all gate types with empty mappings
        for gate_type in gate_counts:
            task4_circuit_mapping[gate_type] = {}

        # For each non-trivial gate type, query LLM with iterative feedback on failure
        for gate_type, template in query_templates.items():
            if gate_type in SKIP_GATE_TYPES:
                continue

            print(f"\n--- Gate type: {gate_type}  (template: {template}) ---")
            messages = [{"role": "user", "content": create_task4_prompt(template, TASK4_TARGET_GATES)}]
            num_trials = 0
            success = False

            while num_trials < TASK4_MAX_TRIALS:
                num_trials += 1
                print(f"  Trial {num_trials}/{TASK4_MAX_TRIALS}")

                # Use full conversation history for context (feedback loop)
                response = openai.chat.completions.create(
                    model=TASK4_MODEL,
                    messages=messages,
                    temperature=0.2,
                    max_tokens=500
                )
                llm_response = response.choices[0].message.content
                messages.append({"role": "assistant", "content": llm_response})

                LLM_circuit = get_circuit_from_response(llm_response)

                if "Incorrect format" in LLM_circuit:
                    print(f"  ⚠️ Format error — sending feedback")
                    feedback = create_task4_format_feedback(template, TASK4_TARGET_GATES)
                    messages.append({"role": "user", "content": feedback})
                    continue

                print(f"  ✅ Mapping: {LLM_circuit.strip()}")
                task4_circuit_mapping[gate_type][target_key] = [LLM_circuit]
                success = True
                break

            task4_gate_trial_counts[gate_type] = num_trials
            if not success:
                print(f"  ❌ No valid mapping after {TASK4_MAX_TRIALS} trials — keeping original gate")

        mapped_count = sum(1 for g in non_trivial_gates if task4_circuit_mapping.get(g))
        avg_trials = sum(task4_gate_trial_counts.values()) / len(task4_gate_trial_counts) if task4_gate_trial_counts else 0
        print(f"\n✅ Got mappings for {mapped_count}/{len(non_trivial_gates)} non-trivial gate types")
        if skipped_gates:
            print(f"   (Skipped by design: {skipped_gates})")
        print(f"Average trials per gate type: {avg_trials:.1f}")

        # Apply all mappings to the full circuit
        print("\n🔄 Applying gate mappings to full circuit...")
        task4_final_circuit = get_mapped_circuit(orig_file_contents, task4_circuit_mapping, target_key)

        # Check functional equivalence on the full mapped circuit
        print("\n🔍 Checking functional equivalence...")
        equiv_result, equiv_details = check_functional_equivalence(
            orig_file_contents, task4_final_circuit, "task4"
        )
        task4_functionally_correct = equiv_result
        print(f"Functional check: {equiv_details}")

        # Evaluate with SIM
        print("\n🔍 Evaluating with SIM detector...")
        task4_sim_score = evaluate_sim(orig_file_contents, task4_final_circuit, "task4")
        task4_evades = task4_sim_score is not None and task4_sim_score < SIM_THRESHOLD

        print(f"\n📊 TASK 4 RESULTS:")
        print(f"Gate types mapped: {mapped_count}/{len(non_trivial_gates)} non-trivial  (skipped: {skipped_gates})")
        print(f"Average trials per gate type: {avg_trials:.1f}")
        print(f"Functional Correctness: {'✅ YES' if task4_functionally_correct else '❌ NO' if task4_functionally_correct is False else '⚠️ UNKNOWN'}")
        print(f"SIM Score: {task4_sim_score:.4f}" if task4_sim_score is not None else "SIM Score: Error")
        print(f"Evades Detection: {'✅ YES' if task4_evades else '❌ NO'}")
        print(f"Model Used: {TASK4_MODEL}")
        print(f"Target Gates: {TASK4_TARGET_GATES}")
        overall_success = task4_functionally_correct and task4_evades
        print(f"Overall Success: {'✅ YES' if overall_success else '❌ NO'}")

        # Show per-gate-type trial counts
        print("\nPer-gate-type trial counts:")
        for gt in gate_counts:
            if gt in SKIP_GATE_TYPES:
                print(f"  {gt}: — (trivial gate, skipped by design)")
            elif gt in task4_gate_trial_counts:
                trials = task4_gate_trial_counts[gt]
                mapped = '✅' if task4_circuit_mapping.get(gt) else '❌'
                print(f"  {gt}: {trials} trial(s) {mapped}")

    except Exception as e:
        print(f"❌ Task 4 failed: {e}")
        task4_circuit_mapping = {}
        task4_final_circuit = None
        task4_sim_score = None
        task4_evades = False
        task4_functionally_correct = None
        task4_gate_trial_counts = {}
else:
    print("⚠️ Skipping Task 4 execution (no API key)")



Gate types to process: ['nand_2', 'xor_2', 'nor_2', 'xnor_2', 'nand_4', 'and_9', 'and_8', 'nand_3']
Skipped (trivial): ['not_1']

Task 4 Configuration:
Target gates: ['NAND']
Model: gpt-4
Max trials per gate type: 5

EXECUTING TASK 4...

--- Gate type: nand_2  (template: Y = NAND(A1, A2)) ---
  Trial 1/5
  ✅ Mapping: Y = NAND(A1, A2)

--- Gate type: xor_2  (template: Y = XOR(A1, A2)) ---
  Trial 1/5
  ✅ Mapping: N1 = NAND(A1, A2)
N2 = NAND(A1, N1)
N3 = NAND(A2, N1)
Y = NAND(N2, N3)

--- Gate type: nor_2  (template: Y = NOR(A1, A2)) ---
  Trial 1/5
  ✅ Mapping: N1 = NAND(A1, A1)
N2 = NAND(A2, A2)
Y = NAND(N1, N2)

--- Gate type: xnor_2  (template: Y = XNOR(A1, A2)) ---
  Trial 1/5
  ✅ Mapping: N1 = NAND(A1, A1)
N2 = NAND(A2, A2)
N3 = NAND(A1, A2)
N4 = NAND(N1, N3)
N5 = NAND(N2, N3)
Y = NAND(N4, N5)

--- Gate type: nand_4  (template: Y = NAND(A1, A2, A3, A4)) ---
  Trial 1/5
  ✅ Mapping: N1 = NAND(A1, A2)
N2 = NAND(A3, A4)
Y = NAND(N1, N2)

--- Gate type: and_9  (template: Y = AND(A1, A2

In [28]:
from IPython.display import display, Markdown

analysis_md = """
## Task 4 Analysis – Divide & Conquer with Iterative Feedback

**Model used:** gpt-4
**Target gates:** NOR
**Gate types mapped:** 8/8 non-trivial gate types
**Average trials per gate type:** 1.0
**SIM Score:** 0.3100
**Success/Failure:** Failure

### Success Rate
The LLM successfully generated mappings for all 8 non-trivial gate types, and every mapping was accepted on the first trial. The trivial gate type `not_1` was skipped by design. This shows that the divide-and-conquer strategy made the gate-level generation process easy for the model, and the output format was stable enough that no retry was actually needed.

However, the final result was still unsuccessful. Even though all gate mappings were generated and parsed correctly, the full reconstructed circuit did not preserve the original functionality.

### SIM Evasion
The SIM score was **0.3100**, which is slightly above the evasion threshold of **0.3**. Therefore, the rewritten circuit did **not** evade SIM-based piracy detection.

This result is very similar to Task 3. The iterative feedback mechanism did not improve evasion, likely because the overall circuit structure remained close to the original design. Rewriting local gates alone was not enough to significantly reduce structural similarity.

### Functional Correctness
The functional equivalence check failed. Yosys reported that the circuits were **not equivalent** and found a SAT counterexample. This means the generated circuit was syntactically valid and fully mapped, but its logic behavior differed from the original circuit.

A likely reason is that some of the gate decompositions were logically incorrect, especially for more complex gates such as `xor_2`, `xnor_2`, `nand_4`, `and_8`, `and_9`, and `nand_3`. Although the mappings looked clean and structurally consistent, at least one or more of them did not implement the exact original Boolean function. Since the final circuit reused those mappings throughout the design, even a small gate-level mistake caused global functional failure.

### Prompt Engineering
Compared to Task 3, Task 4 adds an iterative feedback loop intended to correct either formatting errors or functional mistakes. In this run, however, every gate mapping succeeded on the first attempt, so the feedback mechanism was never meaningfully activated. As a result, Task 4 behaved almost the same as Task 3 in practice.

This suggests that the prompt was strong enough to control formatting, but not strong enough to guarantee logical correctness. For Task 4 to outperform Task 3, the feedback loop needs to catch incorrect gate logic and force the model to revise it, not just accept the first syntactically valid answer.

### Model Comparison
Using GPT-4 improved response quality and consistency at the gate level. All gate mappings were generated quickly, parsed correctly, and accepted without retries. This is better than what would typically be expected from a weaker model in terms of formatting stability.

However, GPT-4 still did not produce a functionally correct full circuit, and it also failed to evade SIM detection. This shows that stronger local generation does not automatically guarantee better end-to-end results. The main limitation is not just model quality, but also the correctness of the reusable gate mappings and the effectiveness of the feedback mechanism.

### Key Insights
Task 4 was expected to improve over Task 3 by adding iterative correction, but in this run it did not produce a better final result. The model generated all mappings successfully on the first try, so the retry mechanism was effectively unused. As a result, Task 4 ended up looking very similar to Task 3 in both SIM score and overall failure.

This experiment shows that iterative feedback only helps when the system can detect and correct mistakes during the mapping stage. If incorrect mappings are accepted too early, then the full circuit still fails equivalence checking. In other words, Task 4 improves robustness in principle, but only if the feedback loop is actually triggered and tied to meaningful verification.

### Summary
Task 4 demonstrated strong formatting stability and fast gate-level generation, but it did not improve the final outcome. The full mapped circuit was not functionally equivalent to the original, and it did not evade SIM-based detection. Therefore, iterative feedback has potential, but in this run it was not effectively utilized, so the performance remained nearly identical to Task 3.
"""

display(Markdown(analysis_md))


## Task 4 Analysis – Divide & Conquer with Iterative Feedback

**Model used:** gpt-4  
**Target gates:** NOR  
**Gate types mapped:** 8/8 non-trivial gate types  
**Average trials per gate type:** 1.0  
**SIM Score:** 0.3100  
**Success/Failure:** Failure  

### Success Rate
The LLM successfully generated mappings for all 8 non-trivial gate types, and every mapping was accepted on the first trial. The trivial gate type `not_1` was skipped by design. This shows that the divide-and-conquer strategy made the gate-level generation process easy for the model, and the output format was stable enough that no retry was actually needed.

However, the final result was still unsuccessful. Even though all gate mappings were generated and parsed correctly, the full reconstructed circuit did not preserve the original functionality.

### SIM Evasion
The SIM score was **0.3100**, which is slightly above the evasion threshold of **0.3**. Therefore, the rewritten circuit did **not** evade SIM-based piracy detection.

This result is very similar to Task 3. The iterative feedback mechanism did not improve evasion, likely because the overall circuit structure remained close to the original design. Rewriting local gates alone was not enough to significantly reduce structural similarity.

### Functional Correctness
The functional equivalence check failed. Yosys reported that the circuits were **not equivalent** and found a SAT counterexample. This means the generated circuit was syntactically valid and fully mapped, but its logic behavior differed from the original circuit.

A likely reason is that some of the gate decompositions were logically incorrect, especially for more complex gates such as `xor_2`, `xnor_2`, `nand_4`, `and_8`, `and_9`, and `nand_3`. Although the mappings looked clean and structurally consistent, at least one or more of them did not implement the exact original Boolean function. Since the final circuit reused those mappings throughout the design, even a small gate-level mistake caused global functional failure.

### Prompt Engineering
Compared to Task 3, Task 4 adds an iterative feedback loop intended to correct either formatting errors or functional mistakes. In this run, however, every gate mapping succeeded on the first attempt, so the feedback mechanism was never meaningfully activated. As a result, Task 4 behaved almost the same as Task 3 in practice.

This suggests that the prompt was strong enough to control formatting, but not strong enough to guarantee logical correctness. For Task 4 to outperform Task 3, the feedback loop needs to catch incorrect gate logic and force the model to revise it, not just accept the first syntactically valid answer.

### Model Comparison
Using GPT-4 improved response quality and consistency at the gate level. All gate mappings were generated quickly, parsed correctly, and accepted without retries. This is better than what would typically be expected from a weaker model in terms of formatting stability.

However, GPT-4 still did not produce a functionally correct full circuit, and it also failed to evade SIM detection. This shows that stronger local generation does not automatically guarantee better end-to-end results. The main limitation is not just model quality, but also the correctness of the reusable gate mappings and the effectiveness of the feedback mechanism.

### Key Insights
Task 4 was expected to improve over Task 3 by adding iterative correction, but in this run it did not produce a better final result. The model generated all mappings successfully on the first try, so the retry mechanism was effectively unused. As a result, Task 4 ended up looking very similar to Task 3 in both SIM score and overall failure.

This experiment shows that iterative feedback only helps when the system can detect and correct mistakes during the mapping stage. If incorrect mappings are accepted too early, then the full circuit still fails equivalence checking. In other words, Task 4 improves robustness in principle, but only if the feedback loop is actually triggered and tied to meaningful verification.

### Summary
Task 4 demonstrated strong formatting stability and fast gate-level generation, but it did not improve the final outcome. The full mapped circuit was not functionally equivalent to the original, and it did not evade SIM-based detection. Therefore, iterative feedback has potential, but in this run it was not effectively utilized, so the performance remained nearly identical to Task 3.


In [29]:

#@title Task 5: Reference Baseline + Strategy Comparison

import pandas as pd
from collections import defaultdict

# ─────────────────────────────────────────────────────────────
# PART A: Reference Baseline — Original Research Cached Mappings
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("PART A: REFERENCE BASELINE")
print("=" * 60)

# TODO: Students can switch LLM to compare baselines
REFERENCE_LLM = 'GPT3dot5'   # Try: 'GPT3dot5', 'GPT4', 'Claude', 'Gemini', 'llama3'
REFERENCE_STRATEGY = 'random'  # 'random' picks best available per gate type

mapping_path = os.path.join(DATA_DIR, 'src', f'cached_circuit_mapping_{REFERENCE_LLM}.pkl')
with open(mapping_path, 'rb') as f:
    reference_mapping = pickle.load(f)

print(f"Loaded cached mapping: {REFERENCE_LLM}")
print(f"\nCoverage for this circuit's gate types:")
for gt in gate_counts:
    strategies = reference_mapping.get(gt, {})
    if strategies:
        print(f"  ✅ {gt}: {list(strategies.keys())}")
    else:
        print(f"  ⬜ {gt}: no cached mapping (gate kept as-is)")

print(f"\n🔄 Applying {REFERENCE_LLM} cached mappings (strategy: {REFERENCE_STRATEGY})...")
ref_circuit = get_mapped_circuit(orig_file_contents, reference_mapping, REFERENCE_STRATEGY)

print("\n🔍 Checking functional equivalence...")
ref_equiv, ref_details = check_functional_equivalence(orig_file_contents, ref_circuit, 'reference')
print(f"Functional check: {ref_details}")

print("\n🔍 Evaluating with SIM detector...")
ref_sim_score = evaluate_sim(orig_file_contents, ref_circuit, 'reference')
ref_evades = ref_sim_score is not None and ref_sim_score < SIM_THRESHOLD

print(f"\n📊 REFERENCE BASELINE RESULTS ({REFERENCE_LLM}):")
print(f"Functional Correctness: {'✅ YES' if ref_equiv else '❌ NO' if ref_equiv is False else '⚠️ UNKNOWN'}")
print(f"SIM Score: {ref_sim_score:.4f}" if ref_sim_score is not None else "SIM Score: Error")
print(f"Evades Detection: {'✅ YES' if ref_evades else '❌ NO'}")

# ─────────────────────────────────────────────────────────────
# PART B: Strategy Comparison
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PART B: COMPREHENSIVE STRATEGY COMPARISON")
print("=" * 60)

all_results = []

# Task 1 Results
if 'task1_result' in locals() and task1_sim_score is not None:
    all_results.append({
        'Task': 'Task 1: Direct Verilog',
        'Method': 'Direct LLM rewriting',
        'SIM_Score': task1_sim_score,
        'Functional': task1_functionally_correct,
        'Overall_Success': bool(task1_functionally_correct and task1_evades),
        'Model': TASK1_MODEL,
        'Notes': 'Single-shot, full circuit'
    })

# Task 2 Results
if 'task2_result' in locals() and task2_sim_score is not None:
    all_results.append({
        'Task': 'Task 2: Boolean Format',
        'Method': 'Boolean intermediate format',
        'SIM_Score': task2_sim_score,
        'Functional': task2_functionally_correct,
        'Overall_Success': bool(task2_functionally_correct and task2_evades),
        'Model': TASK2_MODEL,
        'Notes': 'Via Boolean equations'
    })

# Task 3 Results
if 'task3_final_circuit' in locals() and task3_final_circuit is not None:
    mapped_count3 = sum(1 for g in non_trivial_gates if task3_circuit_mapping.get(g))
    all_results.append({
        'Task': 'Task 3: D&C No Feedback',
        'Method': 'Gate-type mapping, single-shot',
        'SIM_Score': task3_sim_score,
        'Functional': task3_functionally_correct,
        'Overall_Success': bool(task3_functionally_correct and task3_evades),
        'Model': TASK3_MODEL,
        'Notes': f'{mapped_count3}/{len(non_trivial_gates)} gate types mapped'
    })

# Task 4 Results
if 'task4_final_circuit' in locals() and task4_final_circuit is not None:
    mapped_count4 = sum(1 for g in non_trivial_gates if task4_circuit_mapping.get(g))
    avg_trials4 = sum(task4_gate_trial_counts.values()) / len(task4_gate_trial_counts) if task4_gate_trial_counts else 0
    all_results.append({
        'Task': 'Task 4: D&C With Feedback',
        'Method': 'Gate-type mapping, iterative feedback',
        'SIM_Score': task4_sim_score,
        'Functional': task4_functionally_correct,
        'Overall_Success': bool(task4_functionally_correct and task4_evades),
        'Model': TASK4_MODEL,
        'Notes': f'{mapped_count4}/{len(non_trivial_gates)} mapped, {avg_trials4:.1f} avg trials'
    })

# Reference Baseline Results
covered_gates = sum(1 for gt in gate_counts if reference_mapping.get(gt))
all_results.append({
    'Task': f'Reference: {REFERENCE_LLM} Cache',
    'Method': 'Pre-validated research mappings',
    'SIM_Score': ref_sim_score,
    'Functional': ref_equiv,
    'Overall_Success': bool(ref_equiv and ref_evades),
    'Model': REFERENCE_LLM,
    'Notes': f'{covered_gates}/{len(gate_counts)} gate types in cache'
})

# ── Print comparison table ──────────────────────────────────
print("\n📋 SUMMARY TABLE:")
print("-" * 60)
for row in all_results:
    print(f"\n{row['Task']}:")
    print(f"  Method: {row['Method']}")
    print(f"  Model:  {row['Model']}")
    print(f"  SIM Score: {row['SIM_Score']:.4f}" if row['SIM_Score'] is not None else "  SIM Score: N/A")
    func = row['Functional']
    print(f"  Functional: {'✅ YES' if func else ('❌ NO' if func is False else '⚠️ UNKNOWN')}")
    print(f"  Overall Success: {'✅ YES' if row['Overall_Success'] else '❌ NO'}")
    print(f"  Notes: {row['Notes']}")

# ── Ranking ─────────────────────────────────────────────────
print(f"\n🎯 STRATEGY EFFECTIVENESS RANKING:")
print("-" * 40)

sorted_results = sorted(
    all_results,
    key=lambda r: (int(r['Overall_Success']), -(r['SIM_Score'] if r['SIM_Score'] is not None else 1.0))
)[::-1]

for rank, row in enumerate(sorted_results, 1):
    icon = '🏆' if rank == 1 else '🥈' if rank == 2 else '🥉' if rank == 3 else '📍'
    sim_str = f"SIM={row['SIM_Score']:.4f}" if row['SIM_Score'] is not None else "SIM=N/A"
    success_str = '✅' if row['Overall_Success'] else '❌'
    print(f"{rank}. {icon} {row['Task']:35s}  {success_str}  {sim_str}")

# ── Key insights ─────────────────────────────────────────────
print(f"\n📈 KEY INSIGHTS:")
valid_sim = [r for r in all_results if r['SIM_Score'] is not None]
if valid_sim:
    best = min(valid_sim, key=lambda r: r['SIM_Score'])
    print(f"• Lowest SIM score: {best['SIM_Score']:.4f}  ({best['Task']})")
    if ref_sim_score is not None:
        for row in all_results:
            if row['Task'].startswith('Task') and row['SIM_Score'] is not None:
                diff = row['SIM_Score'] - ref_sim_score
                indicator = '🔺 worse' if diff > 0.02 else ('🔻 better' if diff < -0.02 else '≈ similar')
                print(f"• {row['Task']:35s} vs reference: {diff:+.4f} ({indicator})")

model_results = defaultdict(list)
for r in all_results:
    model_results[r['Model']].append(r['Overall_Success'])
if len(model_results) > 1:
    print(f"\n🤖 MODEL COMPARISON:")
    for model, successes in model_results.items():
        rate = sum(successes) / len(successes)
        print(f"• {model}: {rate:.0%} success rate")



PART A: REFERENCE BASELINE
Loaded cached mapping: GPT3dot5

Coverage for this circuit's gate types:
  ✅ nand_2: ['AND_NOT', 'OR_NOT']
  ⬜ not_1: no cached mapping (gate kept as-is)
  ✅ xor_2: ['NAND']
  ✅ nor_2: ['AND_NOT']
  ✅ xnor_2: ['NOR']
  ✅ nand_4: ['NOR', 'AND_NOT', 'OR_NOT']
  ⬜ and_9: no cached mapping (gate kept as-is)
  ⬜ and_8: no cached mapping (gate kept as-is)
  ✅ nand_3: ['AND_NOT', 'OR_NOT']

🔄 Applying GPT3dot5 cached mappings (strategy: random)...

🔍 Checking functional equivalence...
🔍 Running Yosys equivalence check for 'reference'...
Functional check: Circuits formally verified equivalent (Yosys miter+SAT)

🔍 Evaluating with SIM detector...

📊 REFERENCE BASELINE RESULTS (GPT3dot5):
Functional Correctness: ✅ YES
SIM Score: 0.2500
Evades Detection: ✅ YES

PART B: COMPREHENSIVE STRATEGY COMPARISON

📋 SUMMARY TABLE:
------------------------------------------------------------

Task 1: Direct Verilog:
  Method: Direct LLM rewriting
  Model:  gpt-3.5-turbo-16k
  SIM Sc

In [30]:
from IPython.display import display, Markdown

analysis_md = """
## Task 5 Analysis – Strategy Comparison

### Overall Observation
The reference baseline clearly outperforms all student-generated strategies. While the LLM-based approaches either fail functional correctness or SIM evasion, the reference mappings successfully achieve both.

---

### Detailed Analysis

**1. SIM Score Comparison (Task 3/4 vs Reference)**
Task 3 and Task 4 both produced a SIM score of approximately **0.31**, which is higher than the reference baseline (**0.25**). This means the student-generated mappings are more similar to the original circuit and fail to evade detection. In contrast, the reference mappings achieve a lower SIM score while still preserving functionality.

---

**2. Effect of Iterative Feedback (Task 4 vs Task 3)**
The iterative feedback in Task 4 did **not improve results** compared to Task 3. All gate mappings were accepted on the first trial (average trials = 1.0), meaning the feedback loop was never actually triggered. As a result, Task 4 behaved almost identically to Task 3 in both SIM score and functional correctness.

---

**3. Why Student Mappings Differ from Research Mappings**
The student-generated mappings differ from the cached research mappings because they are:
- Generated in a **single pass without verification**
- Not optimized through multiple iterations
- Not validated for functional correctness at the gate level

In contrast, the reference mappings are:
- **Pre-validated**
- Generated using **iterative feedback loops**
- Optimized across multiple models

This ensures both correctness and better structural transformation.

---

**4. Effect of Switching REFERENCE_LLM**
Switching `REFERENCE_LLM` (e.g., to GPT-4, Claude, or Gemini) would change the cached mappings used as the baseline. Different models may produce slightly different mappings, leading to variations in SIM score and possibly structure. However, since all cached mappings are pre-validated, they should still maintain functional correctness and generally perform better than unverified student-generated mappings.

---

**5. Handling and_8 / and_9 (Why Reference Still Fails Here)**
The reference baseline does not include mappings for high-input gates like `and_8` and `and_9`, so these gates remain unchanged. This is because:
- Large fan-in gates are harder to decompose correctly
- They require multi-stage decomposition (tree structure)
- LLMs often generate incorrect or inefficient mappings for them

To fix this, a better approach would be:
- Decompose large AND gates into smaller 2-input gates (e.g., tree of AND2)
- Then map each smaller gate into NOR form
- Or enforce a hierarchical decomposition strategy before mapping

---

### Key Insights

- **SIM score alone is not enough** — correctness is equally important
- **Divide & conquer improves efficiency but not correctness**
- **Feedback only helps if it is actually triggered**
- **Local gate rewriting does not guarantee global correctness**
- **Validated mappings (reference) are critical for success**

---

### Final Conclusion

Task 5 demonstrates that LLM-based circuit rewriting requires more than just generation ability. While different strategies improve aspects such as efficiency or structural variation, none of the student-generated approaches achieved both correctness and evasion.

The reference baseline succeeds because it combines:
- Verified mappings
- Iterative feedback
- Multi-model optimization

This highlights that reliable hardware rewriting requires a **closed-loop system with verification**, rather than single-pass generation.
"""

display(Markdown(analysis_md))


## Task 5 Analysis – Strategy Comparison

### Overall Observation
The reference baseline clearly outperforms all student-generated strategies. While the LLM-based approaches either fail functional correctness or SIM evasion, the reference mappings successfully achieve both.

---

### Detailed Analysis

**1. SIM Score Comparison (Task 3/4 vs Reference)**  
Task 3 and Task 4 both produced a SIM score of approximately **0.31**, which is higher than the reference baseline (**0.25**). This means the student-generated mappings are more similar to the original circuit and fail to evade detection. In contrast, the reference mappings achieve a lower SIM score while still preserving functionality.

---

**2. Effect of Iterative Feedback (Task 4 vs Task 3)**  
The iterative feedback in Task 4 did **not improve results** compared to Task 3. All gate mappings were accepted on the first trial (average trials = 1.0), meaning the feedback loop was never actually triggered. As a result, Task 4 behaved almost identically to Task 3 in both SIM score and functional correctness.

---

**3. Why Student Mappings Differ from Research Mappings**  
The student-generated mappings differ from the cached research mappings because they are:
- Generated in a **single pass without verification**
- Not optimized through multiple iterations
- Not validated for functional correctness at the gate level

In contrast, the reference mappings are:
- **Pre-validated**
- Generated using **iterative feedback loops**
- Optimized across multiple models

This ensures both correctness and better structural transformation.

---

**4. Effect of Switching REFERENCE_LLM**  
Switching `REFERENCE_LLM` (e.g., to GPT-4, Claude, or Gemini) would change the cached mappings used as the baseline. Different models may produce slightly different mappings, leading to variations in SIM score and possibly structure. However, since all cached mappings are pre-validated, they should still maintain functional correctness and generally perform better than unverified student-generated mappings.

---

**5. Handling and_8 / and_9 (Why Reference Still Fails Here)**  
The reference baseline does not include mappings for high-input gates like `and_8` and `and_9`, so these gates remain unchanged. This is because:
- Large fan-in gates are harder to decompose correctly
- They require multi-stage decomposition (tree structure)
- LLMs often generate incorrect or inefficient mappings for them

To fix this, a better approach would be:
- Decompose large AND gates into smaller 2-input gates (e.g., tree of AND2)
- Then map each smaller gate into NOR form
- Or enforce a hierarchical decomposition strategy before mapping

---

### Key Insights

- **SIM score alone is not enough** — correctness is equally important  
- **Divide & conquer improves efficiency but not correctness**  
- **Feedback only helps if it is actually triggered**  
- **Local gate rewriting does not guarantee global correctness**  
- **Validated mappings (reference) are critical for success**

---

### Final Conclusion

Task 5 demonstrates that LLM-based circuit rewriting requires more than just generation ability. While different strategies improve aspects such as efficiency or structural variation, none of the student-generated approaches achieved both correctness and evasion.

The reference baseline succeeds because it combines:
- Verified mappings
- Iterative feedback
- Multi-model optimization

This highlights that reliable hardware rewriting requires a **closed-loop system with verification**, rather than single-pass generation.


In [31]:
from IPython.display import display, Markdown

analysis_md = """
## Final Analysis – Overall Summary (Tasks 1–4)

### 1. Strategy Effectiveness

The reference baseline had the highest success rate, as it was the only approach that achieved both functional correctness and SIM evasion. Among student-generated methods, none achieved full success.

Task 1 achieved the best SIM evasion (SIM ≈ 0.22), but failed due to syntax or functional issues. Task 3 and Task 4 both failed functional correctness and had worse SIM scores (~0.31), meaning they did not evade detection.

The most common errors observed were:
- Syntax errors (especially in Task 1)
- Incorrect gate mappings (Task 3/4)
- Loss of functional equivalence due to incorrect logic decomposition

---

### 2. Model Comparison

GPT-4 did not significantly outperform GPT-3.5 in final outcomes. While GPT-4 produced cleaner and more consistent outputs (especially in Task 3/4), both models failed to achieve functional correctness. This suggests that model quality alone is not sufficient—verification and feedback are equally important.

---

### 3. Target Gate Analysis

In these experiments, NOR-based rewriting did not perform particularly well. NOR is functionally complete, but implementing complex gates (such as XOR or large AND gates) using only NOR requires deep multi-stage logic, which is error-prone.

Certain gates (like AND/OR decompositions) are easier for LLMs because they follow simple and intuitive patterns. In contrast, XOR/XNOR and large fan-in gates (AND_8, AND_9) are much harder due to their more complex Boolean structure.

---

### 4. Feedback Loop Impact

Adding feedback in Task 4 did not significantly improve results compared to Task 3. The average number of trials was 1.0, meaning the feedback loop was never triggered in practice.

As a result, Task 4 behaved almost identically to Task 3. This shows that feedback only helps when errors are detected early and the system forces retries. In this case, incorrect mappings were accepted too early, so feedback had no effect.

---

### 5. Practical Implications

For real IP piracy detection evasion, the most effective approach would be a combination of:
- Structural transformation (to reduce SIM score)
- Verified mappings (to ensure correctness)
- Iterative feedback with validation

The main trade-offs are:
- Task 1: strong obfuscation (low SIM) but poor reliability
- Task 3/4: higher reliability and speed, but poor evasion and correctness
- Reference: best balance, but requires pre-validation and more computation

Defenders could counter these techniques by:
- Using functional equivalence checking instead of only text similarity
- Designing detectors that analyze structural and behavioral patterns
- Combining SIM-based and SAT-based verification methods

---

### Final Summary

**Best strategy overall:**
Reference baseline (cached mappings)

**Key insights:**
- SIM evasion alone is not enough; functional correctness is critical
- Divide-and-conquer improves efficiency but not correctness
- Feedback loops must be actively triggered to be effective

**Surprising findings:**
- Task 1 achieved the best SIM score but still failed overall
- Task 4 did not outperform Task 3 despite having a feedback mechanism

**Recommendations:**
- Use iterative feedback with strict validation at each step
- Decompose large gates into smaller units before mapping
- Combine LLM generation with formal verification tools (e.g., Yosys)
- Avoid relying on single-pass generation for complex hardware rewriting
"""

display(Markdown(analysis_md))


## Final Analysis – Overall Summary (Tasks 1–4)

### 1. Strategy Effectiveness

The reference baseline had the highest success rate, as it was the only approach that achieved both functional correctness and SIM evasion. Among student-generated methods, none achieved full success.

Task 1 achieved the best SIM evasion (SIM ≈ 0.22), but failed due to syntax or functional issues. Task 3 and Task 4 both failed functional correctness and had worse SIM scores (~0.31), meaning they did not evade detection.

The most common errors observed were:
- Syntax errors (especially in Task 1)
- Incorrect gate mappings (Task 3/4)
- Loss of functional equivalence due to incorrect logic decomposition

---

### 2. Model Comparison

GPT-4 did not significantly outperform GPT-3.5 in final outcomes. While GPT-4 produced cleaner and more consistent outputs (especially in Task 3/4), both models failed to achieve functional correctness. This suggests that model quality alone is not sufficient—verification and feedback are equally important.

---

### 3. Target Gate Analysis

In these experiments, NOR-based rewriting did not perform particularly well. NOR is functionally complete, but implementing complex gates (such as XOR or large AND gates) using only NOR requires deep multi-stage logic, which is error-prone.

Certain gates (like AND/OR decompositions) are easier for LLMs because they follow simple and intuitive patterns. In contrast, XOR/XNOR and large fan-in gates (AND_8, AND_9) are much harder due to their more complex Boolean structure.

---

### 4. Feedback Loop Impact

Adding feedback in Task 4 did not significantly improve results compared to Task 3. The average number of trials was 1.0, meaning the feedback loop was never triggered in practice.

As a result, Task 4 behaved almost identically to Task 3. This shows that feedback only helps when errors are detected early and the system forces retries. In this case, incorrect mappings were accepted too early, so feedback had no effect.

---

### 5. Practical Implications

For real IP piracy detection evasion, the most effective approach would be a combination of:
- Structural transformation (to reduce SIM score)
- Verified mappings (to ensure correctness)
- Iterative feedback with validation

The main trade-offs are:
- Task 1: strong obfuscation (low SIM) but poor reliability
- Task 3/4: higher reliability and speed, but poor evasion and correctness
- Reference: best balance, but requires pre-validation and more computation

Defenders could counter these techniques by:
- Using functional equivalence checking instead of only text similarity
- Designing detectors that analyze structural and behavioral patterns
- Combining SIM-based and SAT-based verification methods

---

### Final Summary

**Best strategy overall:**  
Reference baseline (cached mappings)

**Key insights:**  
- SIM evasion alone is not enough; functional correctness is critical  
- Divide-and-conquer improves efficiency but not correctness  
- Feedback loops must be actively triggered to be effective  

**Surprising findings:**  
- Task 1 achieved the best SIM score but still failed overall  
- Task 4 did not outperform Task 3 despite having a feedback mechanism  

**Recommendations:**  
- Use iterative feedback with strict validation at each step  
- Decompose large gates into smaller units before mapping  
- Combine LLM generation with formal verification tools (e.g., Yosys)  
- Avoid relying on single-pass generation for complex hardware rewriting  
